# SciML Autoresearch — Colab / CUDA Notebook

PyTorch port of the MLX training harness for NVIDIA GPU (Colab).  
Results land in `results.json` and are perfectly compatible with your local dashboard.

## Quick-start
1. **Colab**: Runtime → Change runtime type → **T4 / A100 GPU**.
2. **Setup Repo**: Set `REPO_PATH` in Cell 2. Recommended: mount Google Drive or `git clone` into `/content`.
3. **Run Experiments**: Pick a benchmark in the **Run Experiments** section.
4. **Sync Results**: After training, call `git_commit_results()` (Cell 13) and `git push` to see results in your local dashboard.

---

## Cell 1 — Environment detection & package install

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Running in Google Colab')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'matplotlib', 'tqdm', 'pyyaml'], check=True)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

### 💡 VS Code Colab Extension Tips

1. **Mount Drive**: Press `Cmd+Shift+P` → `Colab: Mount Google Drive to Server...`. This ensures your `results.json` and `figs/` are saved back to your Google Drive and synced to your Mac.
2. **Auto-Save**: VS Code will automatically sync cell outputs back to this local file.
3. **Terminal**: You can use the VS Code terminal to run `git` commands on the remote server if you prefer Git over Drive.

## Cell 2 — Repo path & results directory setup

In [ ]:
from pathlib import Path
import json, csv, time, uuid, math

# ── Configure these paths ─────────────────────────────────────────────────────
# REMOTE_REPO_PATH: the path on the Colab server (remote)
# LOCAL_REPO_PATH:  the path on your Mac

if IN_COLAB:
    # If using 'Colab: Mount Google Drive' in VS Code, we check Drive first
    DRIVE_PATH = Path('/content/drive/MyDrive/autoresearch-mlx')
    DEFAULT_PATH = Path('/content/autoresearch-mlx')
    
    if DRIVE_PATH.exists():
        REPO_PATH = DRIVE_PATH
        print(f'Detected repo in Google Drive: {REPO_PATH}')
    else:
        REPO_PATH = DEFAULT_PATH
        if not REPO_PATH.exists():
            print(f'Repo not found in Drive. Using default: {REPO_PATH}')
            # REPO_PATH.mkdir(parents=True, exist_ok=True)
else:
    # Running locally on M1
    REPO_PATH = Path.cwd()

LOGS_DIR     = REPO_PATH / 'logs'
FIGS_DIR     = REPO_PATH / 'figs'
RESULTS_JSON = REPO_PATH / 'results.json'
RESULTS_TSV  = REPO_PATH / 'results.tsv'
CACHE_DIR    = Path.home() / '.cache' / 'sciml_autoresearch'

LOGS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo:    {REPO_PATH}')
print(f'Results: {RESULTS_JSON}')


## Cell 3 — ExperimentConfig (identical to experiments.py)

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class ExperimentConfig:
    name:        str
    benchmark:   str
    model:       str
    hidden_dim:  int
    n_layers:    int
    n_modes:     int   = 16
    n_levels:    int   = 3
    n_head:      int   = 4
    slice_num:   int   = 32
    lr:          float = 1e-3
    batch_size:  int   = 64
    grad_clip:   float = 1.0
    loss_type:   str   = 'l2_rel'
    h1_alpha:    float = 0.1
    augment:     bool  = False
    budget_s:    int   = 300
    parent_name: str   = ''
    priority:    int   = 5
    rationale:   str   = ''
    expected:    str   = ''
    paper_ref:   str   = ''

print('ExperimentConfig ready')

## Cell 4 — PDE data generation (numpy, no framework dependency)

In [ ]:
import numpy as np

# ── Fixed constants (must match prepare.py) ───────────────────────────────────
GRID_SIZE   = 64
T_FINAL     = 1.0
NU          = 0.01 / math.pi
N_TRAIN     = 4096
N_VAL       = 256
TRAIN_SEED  = 7
VAL_SEED    = 42
EVAL_BATCH  = 64

# ── IC generators ────────────────────────────────────────────────────────────
def _random_ic(n, N, rng, n_modes=10):
    k      = np.arange(1, n_modes + 1, dtype=np.float64)
    decay  = k ** -1.5
    cos_c  = rng.randn(n, n_modes) * decay
    sin_c  = rng.randn(n, n_modes) * decay
    x      = 2.0 * np.pi * np.arange(N, dtype=np.float64) / N
    angles = k[:, None] * x[None, :]
    u0     = cos_c @ np.cos(angles) + sin_c @ np.sin(angles)
    return u0.astype(np.float32)

def _random_ic_2d(n, N, rng, n_modes=5, scale=0.1, offset=1.0):
    x, y   = np.linspace(0, 1, N), np.linspace(0, 1, N)
    X, Y   = np.meshgrid(x, y)
    u0     = np.full((n, N, N), offset, dtype=np.float64)
    for i in range(n):
        for _ in range(n_modes):
            amp     = rng.randn() * scale
            kx, ky  = rng.randint(1, 5, size=2)
            u0[i]  += amp * np.sin(2 * np.pi * (kx * X + ky * Y))
    return u0.astype(np.float32)

# ── 1D PDE Solvers ────────────────────────────────────────────────────────────
def solve_burgers_batch(u0, nu=NU, T=T_FINAL, n_steps=500):
    _, N   = u0.shape
    k      = np.fft.rfftfreq(N, d=1.0 / N)
    dt     = T / n_steps
    impl   = 1.0 / (1.0 + nu * k ** 2 * dt)
    ik     = 1j * k
    cutoff = N // 3
    u_hat  = np.fft.rfft(u0.astype(np.float64), axis=1)
    for _ in range(n_steps):
        uh_d           = u_hat.copy()
        uh_d[:, cutoff:] = 0.0
        u_phys  = np.fft.irfft(uh_d, n=N, axis=1)
        ux_phys = np.fft.irfft(ik * uh_d, n=N, axis=1).real
        nonlin  = np.fft.rfft(-u_phys * ux_phys, axis=1)
        u_hat   = impl * (u_hat + dt * nonlin)
    return np.fft.irfft(u_hat, n=N, axis=1).astype(np.float32)

def solve_kdv_batch(u0, T=1.0, n_steps=1000):
    _, N   = u0.shape
    k      = np.fft.rfftfreq(N, d=1.0 / N)
    ik     = 1j * k
    ik3    = (1j * k) ** 3
    cutoff = N // 3
    dt     = T / n_steps
    L      = -ik3
    E      = np.exp(L * dt)
    E2     = np.exp(L * dt / 2.0)
    u_hat  = np.fft.rfft(u0.astype(np.float64), axis=1)
    def nonlin(uh):
        uhd           = uh.copy()
        uhd[:, cutoff:] = 0.0
        u_phys  = np.fft.irfft(uhd, n=N, axis=1)
        ux_phys = np.fft.irfft(ik * uhd, n=N, axis=1).real
        return np.fft.rfft(-u_phys * ux_phys, axis=1)
    for _ in range(n_steps):
        N0 = nonlin(u_hat)
        a  = E2 * u_hat + E2 * dt / 2.0 * N0
        Na = nonlin(a)
        b  = E2 * u_hat + E2 * dt / 2.0 * Na
        Nb = nonlin(b)
        c  = E2 * a     + E2 * dt / 2.0 * (2.0 * Nb - N0)
        Nc = nonlin(c)
        u_hat = E * u_hat + dt / 6.0 * (E * N0 + 2.0 * E2 * (Na + Nb) + Nc)
    return np.fft.irfft(u_hat, n=N, axis=1).astype(np.float32)

def solve_wave_batch(u0, ut0, c=1.0, T=1.0, n_steps=400):
    _, N   = u0.shape
    k      = np.fft.rfftfreq(N, d=1.0 / N)
    omega2 = (c * k) ** 2
    dt     = T / n_steps
    u_hat  = np.fft.rfft(u0.astype(np.float64),  axis=1)
    ut_hat = np.fft.rfft(ut0.astype(np.float64), axis=1)
    for _ in range(n_steps):
        ut_hat -= 0.5 * dt * omega2 * u_hat
        u_hat  += dt * ut_hat
        ut_hat -= 0.5 * dt * omega2 * u_hat
    return np.fft.irfft(u_hat, n=N, axis=1).astype(np.float32)

# ── 2D NS fix solver ─────────────────────────────────────────────────────────
def solve_ns_fix_batch(w0, nu=1e-2, T=1.0, n_steps=1000):
    _, N, _ = w0.shape
    dt      = T / n_steps
    k_int   = np.fft.fftfreq(N, d=1.0 / N)
    k1, k2  = np.meshgrid(k_int, k_int)
    lap     = -(k1**2 + k2**2)
    lap[0, 0] = 1.0
    cutoff  = (2 * N) // 3
    w_hat   = np.fft.fft2(w0.astype(np.float64), axes=(1, 2))
    for _ in range(n_steps):
        wh_d  = w_hat.copy()
        mask  = (np.abs(k1 * N) > cutoff) | (np.abs(k2 * N) > cutoff)
        wh_d[:, mask] = 0.0
        psi   = wh_d / lap[None]
        psi[:, 0, 0] = 0.0
        u     = np.fft.ifft2(1j * k2[None] * psi, axes=(1, 2)).real
        v     = np.fft.ifft2(-1j * k1[None] * psi, axes=(1, 2)).real
        wx    = np.fft.ifft2(1j * k1[None] * wh_d, axes=(1, 2)).real
        wy    = np.fft.ifft2(1j * k2[None] * wh_d, axes=(1, 2)).real
        nonlin = np.fft.fft2(u * wx + v * wy, axes=(1, 2))
        w_hat  = (w_hat - dt * nonlin) / (1.0 - dt * nu * lap[None])
        if np.any(np.isnan(w_hat)):
            break
    return np.fft.ifft2(w_hat, axes=(1, 2)).real.astype(np.float32)

# ── Darcy 2D fix (PCG) ───────────────────────────────────────────────────────
def solve_darcy_fix_batch(a, f, n_iter=40):
    B, N, _ = a.shape
    a_d, f_d = a.astype(np.float64), f.astype(np.float64)
    k_int = np.fft.fftfreq(N, d=1.0 / N)
    kx, ky = np.meshgrid(2 * np.pi * k_int, 2 * np.pi * k_int)
    lap_pos = kx**2 + ky**2
    lap_pos[0, 0] = 1.0
    a_mean = a_d.mean(axis=(1, 2), keepdims=True)
    def apply_A(v):
        v_hat = np.fft.fft2(v, axes=(1, 2))
        vx = np.fft.ifft2(1j * kx[None] * v_hat, axes=(1, 2)).real
        vy = np.fft.ifft2(1j * ky[None] * v_hat, axes=(1, 2)).real
        Av = -np.fft.ifft2(
            1j * kx[None] * np.fft.fft2(a_d * vx, axes=(1, 2))
            + 1j * ky[None] * np.fft.fft2(a_d * vy, axes=(1, 2)), axes=(1, 2)).real
        return Av
    def apply_P_inv(r):
        r_hat = np.fft.fft2(r, axes=(1, 2))
        Pr = np.fft.ifft2(r_hat / (a_mean * lap_pos[None]), axes=(1, 2)).real
        Pr -= Pr.mean(axis=(1, 2), keepdims=True)
        return Pr
    u = np.zeros((B, N, N), dtype=np.float64)
    r = f_d - apply_A(u)
    r -= r.mean(axis=(1, 2), keepdims=True)
    z = apply_P_inv(r)
    p = z.copy()
    rz = (r * z).sum(axis=(1, 2), keepdims=True)
    for _ in range(n_iter):
        Ap   = apply_A(p)
        pAp  = (p * Ap).sum(axis=(1, 2), keepdims=True)
        alpha = rz / (pAp + 1e-30)
        u   += alpha * p
        r   -= alpha * Ap
        r   -= r.mean(axis=(1, 2), keepdims=True)
        z    = apply_P_inv(r)
        rz_new = (r * z).sum(axis=(1, 2), keepdims=True)
        beta = rz_new / (rz + 1e-30)
        p    = z + beta * p
        rz   = rz_new
    return u.astype(np.float32)

print('PDE solvers ready')

## Cell 5 — Dataset generation & caching

In [ ]:
from tqdm.auto import tqdm

_DATA_CACHE = {}

def _gen_dataset(benchmark, n, seed):
    rng = np.random.RandomState(seed)
    print(f'  Generating {benchmark} n={n} seed={seed}...')
    t0 = time.time()
    if benchmark == 'burgers_1d':
        x = _random_ic(n, GRID_SIZE, rng)
        y = solve_burgers_batch(x)
    elif benchmark == 'kdv_1d':
        x = _random_ic(n, GRID_SIZE, rng)
        y = solve_kdv_batch(x)
    elif benchmark == 'wave_1d':
        x   = _random_ic(n, GRID_SIZE, rng)
        ut0 = _random_ic(n, GRID_SIZE, rng)
        y   = solve_wave_batch(x, ut0)
    elif benchmark == 'darcy_2d':
        rng_f = np.random.RandomState(seed + 1000)
        x = _random_ic_2d(n, GRID_SIZE, rng, scale=0.1, offset=1.0)
        f = _random_ic_2d(n, GRID_SIZE, rng_f, scale=1.0, offset=0.0)
        y = solve_darcy_fix_batch(x, f)
    elif benchmark == 'ns_2d':
        x = _random_ic_2d(n, GRID_SIZE, rng, scale=0.1, offset=0.0)  # CFL-safe amplitude
        y = solve_ns_fix_batch(x)
    else:
        raise ValueError(f'Unknown benchmark: {benchmark}')
    print(f'  Done in {time.time()-t0:.1f}s')
    return x, y

def load_data(benchmark, split):
    """Load or generate dataset, with disk + memory caching."""
    key = (benchmark, split)
    if key in _DATA_CACHE:
        return _DATA_CACHE[key]

    n    = N_VAL if split == 'val' else N_TRAIN
    seed = VAL_SEED if split == 'val' else TRAIN_SEED

    # Extended benchmarks use a different cache naming convention
    ext_benchmarks = {'kdv_1d', 'wave_1d', 'darcy_2d', 'ns_2d'}
    if benchmark in ext_benchmarks:
        tag = f'N{n}_ext'
    else:
        tag = f'N{n}'
    cache_file = CACHE_DIR / f'{benchmark}_{split}_{tag}.npz'

    if cache_file.exists():
        d = np.load(cache_file)
        x, y = d['inputs'], d['targets']
        print(f'Loaded {split} cache: {cache_file.name}  ({len(x)} samples)')
    else:
        x, y = _gen_dataset(benchmark, n, seed)
        np.savez(cache_file, inputs=x, targets=y)
        print(f'Cached → {cache_file}')

    _DATA_CACHE[key] = (x, y)
    return x, y


def make_torch_loader(benchmark, split, batch_size, device):
    """Infinite PyTorch tensor iterator."""
    x_np, y_np = load_data(benchmark, split)
    x_t = torch.from_numpy(x_np).to(device)
    y_t = torch.from_numpy(y_np).to(device)
    n   = len(x_t)
    rng = np.random.RandomState(99)

    if split == 'val':
        i = 0
        while True:
            end = min(i + batch_size, n)
            yield x_t[i:end], y_t[i:end]
            i = end
            if i >= n:
                i = 0
    else:
        while True:
            perm = rng.permutation(n)
            for i in range(0, n - batch_size + 1, batch_size):
                idx = perm[i : i + batch_size]
                yield x_t[idx], y_t[idx]

print('Data loaders ready')

## Cell 6 — PyTorch model implementations

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── 1D Spectral Convolution ───────────────────────────────────────────────────
class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, n_modes):
        super().__init__()
        self.in_ch   = in_ch
        self.out_ch  = out_ch
        self.n_modes = n_modes
        scale = (in_ch * out_ch) ** -0.5
        self.wr = nn.Parameter(torch.randn(n_modes, in_ch, out_ch) * scale)
        self.wi = nn.Parameter(torch.randn(n_modes, in_ch, out_ch) * scale)

    def forward(self, x):          # x: [B, N, C]
        B, N, _ = x.shape
        x_ft = torch.fft.rfft(x, dim=1)                   # [B, N//2+1, C] complex
        xr   = x_ft[:, :self.n_modes, :].real
        xi   = x_ft[:, :self.n_modes, :].imag
        out_r = (torch.einsum('bmi,mio->bmo', xr, self.wr)
               - torch.einsum('bmi,mio->bmo', xi, self.wi))
        out_i = (torch.einsum('bmi,mio->bmo', xr, self.wi)
               + torch.einsum('bmi,mio->bmo', xi, self.wr))
        out_modes = torch.complex(out_r, out_i)
        n_rfft = N // 2 + 1
        if n_rfft > self.n_modes:
            pad    = torch.zeros(B, n_rfft - self.n_modes, self.out_ch,
                                 dtype=torch.complex64, device=x.device)
            out_ft = torch.cat([out_modes, pad], dim=1)
        else:
            out_ft = out_modes
        return torch.fft.irfft(out_ft, n=N, dim=1)         # [B, N, out_ch]


# ── FNO Block ────────────────────────────────────────────────────────────────
class FNOBlock1d(nn.Module):
    def __init__(self, channels, n_modes):
        super().__init__()
        self.spec = SpectralConv1d(channels, channels, n_modes)
        self.w    = nn.Linear(channels, channels)

    def forward(self, x):
        return F.gelu(self.spec(x) + self.w(x))


# ── FNO1d ─────────────────────────────────────────────────────────────────────
class FNO1d(nn.Module):
    def __init__(self, n_modes, hidden_dim, n_layers, in_ch=2):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock1d(hidden_dim, n_modes) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):          # u0: [B, N]
        B, N = u0.shape
        grid = torch.linspace(0, 1, N, device=u0.device).unsqueeze(0).expand(B, -1)
        x    = torch.stack([u0, grid], dim=-1)             # [B, N, 2]
        x    = self.lift(x)
        for blk in self.blocks:
            x = blk(x)
        x = F.gelu(self.proj1(x))
        return self.proj2(x).squeeze(-1)                   # [B, N]


# ── RFNO1d (Pre-LN Residual FNO) ─────────────────────────────────────────────
class FNOBlockResidual1d(nn.Module):
    def __init__(self, channels, n_modes):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.spec = SpectralConv1d(channels, channels, n_modes)
        self.w    = nn.Linear(channels, channels)

    def forward(self, x):
        h = self.norm(x)
        return x + F.gelu(self.spec(h) + self.w(h))


class RFNO1d(nn.Module):
    def __init__(self, n_modes, hidden_dim, n_layers, in_ch=2):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlockResidual1d(hidden_dim, n_modes) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(hidden_dim)
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = torch.linspace(0, 1, N, device=u0.device).unsqueeze(0).expand(B, -1)
        x    = torch.stack([u0, grid], dim=-1)
        x    = self.lift(x)
        for blk in self.blocks:
            x = blk(x)
        x = F.gelu(self.proj1(self.norm(x)))
        return self.proj2(x).squeeze(-1)


# ── 2D Spectral Convolution ───────────────────────────────────────────────────
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, n_modes1, n_modes2):
        super().__init__()
        self.n_modes1, self.n_modes2 = n_modes1, n_modes2
        self.out_ch = out_ch
        scale = (in_ch * out_ch) ** -0.5
        self.wr1 = nn.Parameter(torch.randn(n_modes1, n_modes2, in_ch, out_ch) * scale)
        self.wi1 = nn.Parameter(torch.randn(n_modes1, n_modes2, in_ch, out_ch) * scale)
        self.wr2 = nn.Parameter(torch.randn(n_modes1, n_modes2, in_ch, out_ch) * scale)
        self.wi2 = nn.Parameter(torch.randn(n_modes1, n_modes2, in_ch, out_ch) * scale)

    def _cmul(self, xr, xi, wr, wi):
        return (torch.einsum('bmki,mkio->bmko', xr, wr)
              - torch.einsum('bmki,mkio->bmko', xi, wi),
                torch.einsum('bmki,mkio->bmko', xr, wi)
              + torch.einsum('bmki,mkio->bmko', xi, wr))

    def forward(self, x):           # x: [B, N1, N2, C]
        B, N1, N2, C = x.shape
        x_ft   = torch.fft.rfft2(x.permute(0, 3, 1, 2))   # [B, C, N1, N2//2+1]
        x_ft   = x_ft.permute(0, 2, 3, 1)                  # [B, N1, N2//2+1, C]
        out_ft = torch.zeros(B, N1, N2 // 2 + 1, self.out_ch,
                             dtype=torch.complex64, device=x.device)
        m1, m2 = self.n_modes1, self.n_modes2
        xr1, xi1 = x_ft[:, :m1, :m2, :].real, x_ft[:, :m1, :m2, :].imag
        r1, i1   = self._cmul(xr1, xi1, self.wr1, self.wi1)
        out_ft[:, :m1, :m2, :] = torch.complex(r1, i1)
        xr2, xi2 = x_ft[:, -m1:, :m2, :].real, x_ft[:, -m1:, :m2, :].imag
        r2, i2   = self._cmul(xr2, xi2, self.wr2, self.wi2)
        out_ft[:, -m1:, :m2, :] = torch.complex(r2, i2)
        # irfft2 expects [B, C, N1, N2]
        out = torch.fft.irfft2(out_ft.permute(0, 3, 1, 2), s=(N1, N2))
        return out.permute(0, 2, 3, 1)                      # [B, N1, N2, out_ch]


class FNOBlock2d(nn.Module):
    def __init__(self, channels, n_modes1, n_modes2):
        super().__init__()
        self.spec = SpectralConv2d(channels, channels, n_modes1, n_modes2)
        self.w    = nn.Linear(channels, channels)

    def forward(self, x):
        return F.gelu(self.spec(x) + self.w(x))


class FNO2d(nn.Module):
    def __init__(self, n_modes1, n_modes2, hidden_dim, n_layers, in_ch=3):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock2d(hidden_dim, n_modes1, n_modes2) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):          # u0: [B, N1, N2]
        B, N1, N2 = u0.shape
        g1 = torch.linspace(0, 1, N1, device=u0.device)[None, :, None].expand(B, -1, N2)
        g2 = torch.linspace(0, 1, N2, device=u0.device)[None, None, :].expand(B, N1, -1)
        x  = torch.stack([u0, g1, g2], dim=-1)              # [B, N1, N2, 3]
        x  = self.lift(x)
        for blk in self.blocks:
            x = blk(x)
        x = F.gelu(self.proj1(x))
        return self.proj2(x).squeeze(-1)                     # [B, N1, N2]


# ── UNO1d (U-shaped Neural Operator) ─────────────────────────────────────────
class UNO1d(nn.Module):
    def __init__(self, n_modes, hidden_dim, n_layers=2, in_ch=2):
        super().__init__()
        h, m = hidden_dim, n_modes
        self.lift  = nn.Linear(in_ch, h)
        self.enc0  = nn.Sequential(*[FNOBlock1d(h, m) for _ in range(n_layers)])
        self.down0 = nn.Linear(h, 2 * h)
        self.enc1  = nn.Sequential(*[FNOBlock1d(2 * h, max(m // 2, 1)) for _ in range(n_layers)])
        self.down1 = nn.Linear(2 * h, 4 * h)
        self.bot   = nn.Sequential(*[FNOBlock1d(4 * h, max(m // 4, 1)) for _ in range(n_layers)])
        self.up1   = nn.Linear(4 * h + 2 * h, 2 * h)
        self.dec1  = nn.Sequential(*[FNOBlock1d(2 * h, max(m // 2, 1)) for _ in range(n_layers)])
        self.up0   = nn.Linear(2 * h + h, h)
        self.dec0  = nn.Sequential(*[FNOBlock1d(h, m) for _ in range(n_layers)])
        self.proj1 = nn.Linear(h, h // 2)
        self.proj2 = nn.Linear(h // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = torch.linspace(0, 1, N, device=u0.device).unsqueeze(0).expand(B, -1)
        x    = torch.stack([u0, grid], dim=-1)
        x    = self.lift(x)
        s0   = self.enc0(x)
        x    = self.down0(s0)[:, ::2, :]
        s1   = self.enc1(x)
        x    = self.down1(s1)[:, ::2, :]
        x    = self.bot(x)
        x    = x.repeat_interleave(2, dim=1)
        x    = self.up1(torch.cat([x, s1], dim=-1))
        x    = self.dec1(x)
        x    = x.repeat_interleave(2, dim=1)
        x    = self.up0(torch.cat([x, s0], dim=-1))
        x    = self.dec0(x)
        x    = F.gelu(self.proj1(x))
        return self.proj2(x).squeeze(-1)


# ── Model factory ─────────────────────────────────────────────────────────────
BENCHMARK_DIM = {
    'burgers_1d': '1d', 'kdv_1d': '1d', 'wave_1d': '1d',
    'darcy_2d': '2d', 'ns_2d': '2d',
}

def build_model(cfg: ExperimentConfig, device) -> nn.Module:
    dim = BENCHMARK_DIM.get(cfg.benchmark, '1d')
    m   = cfg.model.upper()
    if m == 'FNO' and dim == '1d':
        model = FNO1d(cfg.n_modes, cfg.hidden_dim, cfg.n_layers)
    elif m == 'RFNO' and dim == '1d':
        model = RFNO1d(cfg.n_modes, cfg.hidden_dim, cfg.n_layers)
    elif m == 'UNO' and dim == '1d':
        model = UNO1d(cfg.n_modes, cfg.hidden_dim, cfg.n_layers)
    elif m in ('FNO', 'FNO2D') and dim == '2d':
        model = FNO2d(cfg.n_modes, cfg.n_modes, cfg.hidden_dim, cfg.n_layers)
    else:
        raise ValueError(f'Unsupported model={cfg.model} for benchmark={cfg.benchmark}')
    return model.to(device)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print('Models ready: FNO1d, RFNO1d, UNO1d, FNO2d')

## Cell 6.5 — Extended PyTorch Model Registry

PyTorch ports of all remaining MLX-only models:
S4DLayer · SSNO · MemNO · MambaNO · ChebyKAN · GNOT · Transolver · SNO2D ·
FEDONet2D · HANO2D · VSMNO2D · HybridFNO · HybridDecoder · WNO · AFNO · TFNO ·
HamiltonianNO · EnergyFNO · NeuralODE · TimeDeepONet · DeepONet · PACMANN · PINN

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────────────
# Grid helpers
# ─────────────────────────────────────────────────────────────────────────────
def _grid1d(B, N, device):
    return torch.linspace(0, 1, N, device=device).unsqueeze(0).expand(B, -1)

def _grid2d(B, N1, N2, device):
    g1 = torch.linspace(0, 1, N1, device=device)[None, :, None].expand(B, -1, N2)
    g2 = torch.linspace(0, 1, N2, device=device)[None, None, :].expand(B, N1, -1)
    return g1, g2

def _lift2d(x):
    """Ensure x is [B, N1, N2, 1] and append 2-D grid."""
    if x.ndim == 3:
        B, N1, N2 = x.shape
        x = x.unsqueeze(-1)
    else:
        B, N1, N2 = x.shape[:3]
    g1, g2 = _grid2d(B, N1, N2, x.device)
    return torch.cat([x, g1.unsqueeze(-1), g2.unsqueeze(-1)], dim=-1), B, N1, N2


# ─────────────────────────────────────────────────────────────────────────────
# S4D Layer (Diagonal State Space Model)
# ─────────────────────────────────────────────────────────────────────────────
class S4DLayer(nn.Module):
    """Diagonal S4 — FFT-based convolution kernel, O(L log L)."""
    def __init__(self, d_model, d_state=32):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.log_dt  = nn.Parameter(torch.FloatTensor(d_model).uniform_(math.log(1e-3), math.log(0.1)))
        self.register_buffer('a_real', torch.full((d_model, d_state), -0.5))
        im = torch.arange(d_state, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.register_buffer('a_imag', im.clone())
        s = d_state ** -0.5
        self.b_real = nn.Parameter(torch.randn(d_model, d_state) * s)
        self.b_imag = nn.Parameter(torch.randn(d_model, d_state) * s)
        self.c_real = nn.Parameter(torch.randn(d_model, d_state) * s)
        self.c_imag = nn.Parameter(torch.randn(d_model, d_state) * s)
        self.d_skip = nn.Parameter(torch.ones(d_model))

    def _kernel(self, L):
        dt = self.log_dt.exp()
        t  = torch.arange(L, device=dt.device, dtype=torch.float32)
        at_r = self.a_real[:, None, :] * dt[:, None, None] * t[None, :, None]
        at_i = self.a_imag[:, None, :] * dt[:, None, None] * t[None, :, None]
        exp_r = at_r.exp()
        cb_rr = (self.c_real * self.b_real)[:, None, :]
        cb_ii = (self.c_imag * self.b_imag)[:, None, :]
        cb_ri = (self.c_real * self.b_imag)[:, None, :]
        cb_ir = (self.c_imag * self.b_real)[:, None, :]
        k = ((cb_rr - cb_ii) * at_i.cos() + (cb_ri + cb_ir) * at_i.sin()) * exp_r
        return k.sum(-1)  # [H, L]

    def forward(self, x):  # [B, L, H]
        B, L, H = x.shape
        k  = self._kernel(L)
        xf = torch.fft.rfft(x.permute(0, 2, 1), n=2 * L)
        kf = torch.fft.rfft(k, n=2 * L)
        y  = torch.fft.irfft(xf * kf[None], n=2 * L)[..., :L].permute(0, 2, 1)
        return y + x * self.d_skip[None, None, :]


# ─────────────────────────────────────────────────────────────────────────────
# SSNO — State-Space Neural Operator (1-D)
# ─────────────────────────────────────────────────────────────────────────────
class _SSNOBlock1d(nn.Module):
    def __init__(self, hidden_dim, n_modes, d_state=32):
        super().__init__()
        self.norm     = nn.LayerNorm(hidden_dim)
        self.ssm      = S4DLayer(hidden_dim, d_state)
        self.ssm_proj = nn.Linear(hidden_dim, hidden_dim)
        self.wr       = nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.wi       = nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.n_modes  = n_modes
        self.spec_proj= nn.Linear(hidden_dim, hidden_dim)
        self.W        = nn.Linear(hidden_dim, hidden_dim)
        self.gate     = nn.Linear(hidden_dim * 2, hidden_dim)

    def _spec(self, x):
        B, N, H = x.shape
        xf  = torch.fft.rfft(x, dim=1)
        m   = min(self.n_modes, xf.shape[1])
        xr, xi = xf[:, :m, :].real, xf[:, :m, :].imag
        or_ = torch.einsum('bmc,mco->bmo', xr, self.wr[:m]) - torch.einsum('bmc,mco->bmo', xi, self.wi[:m])
        oi  = torch.einsum('bmc,mco->bmo', xr, self.wi[:m]) + torch.einsum('bmc,mco->bmo', xi, self.wr[:m])
        out = torch.complex(or_, oi)
        pad = xf.shape[1] - m
        if pad > 0:
            out = torch.cat([out, torch.zeros(B, pad, H, dtype=out.dtype, device=x.device)], 1)
        return torch.fft.irfft(out, n=N, dim=1)

    def forward(self, x):
        h    = self.norm(x)
        ssm  = F.gelu(self.ssm_proj(self.ssm(h)))
        spec = F.gelu(self.spec_proj(self._spec(h)))
        fuse = F.gelu(self.gate(torch.cat([ssm, spec], -1)))
        return x + fuse + self.W(h)

class SSNO(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, n_modes=16, d_state=32, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([_SSNOBlock1d(hidden_dim, n_modes, d_state) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# MemNO — Memory Neural Operator (S4D memory + FNO)
# ─────────────────────────────────────────────────────────────────────────────
class _MemNOBlock1d(nn.Module):
    def __init__(self, hidden_dim, n_modes, d_state=32):
        super().__init__()
        self.norm    = nn.LayerNorm(hidden_dim)
        self.memory  = S4DLayer(hidden_dim, d_state)
        self.wr      = nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.wi      = nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.n_modes = n_modes
        self.W       = nn.Linear(hidden_dim, hidden_dim)

    def _spec(self, x):
        B, N, H = x.shape
        xf = torch.fft.rfft(x, dim=1)
        m  = min(self.n_modes, xf.shape[1])
        xr, xi = xf[:, :m].real, xf[:, :m].imag
        or_ = torch.einsum('bmc,mco->bmo', xr, self.wr[:m]) - torch.einsum('bmc,mco->bmo', xi, self.wi[:m])
        oi  = torch.einsum('bmc,mco->bmo', xr, self.wi[:m]) + torch.einsum('bmc,mco->bmo', xi, self.wr[:m])
        out = torch.complex(or_, oi)
        if xf.shape[1] > m:
            out = torch.cat([out, torch.zeros(B, xf.shape[1]-m, H, dtype=out.dtype, device=x.device)], 1)
        return torch.fft.irfft(out, n=N, dim=1)

    def forward(self, x):
        h = self.norm(x)
        return x + F.gelu(self._spec(h) + self.memory(h) + self.W(h))

class MemNO(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, n_modes=16, d_state=32, in_ch=2, **kw):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        self.blocks = nn.ModuleList([_MemNOBlock1d(hidden_dim, n_modes, d_state) for _ in range(n_layers)])
        self.proj   = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Linear(hidden_dim // 2, 1))

    def forward(self, u0):
        B, N = u0.shape[0], u0.shape[1] if u0.ndim == 2 else u0.numel() // u0.shape[0]
        u0f  = u0.reshape(B, N)
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0f, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj(x).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# MambaNO — Mamba-style SSM + spectral filter (1-D)
# ─────────────────────────────────────────────────────────────────────────────
class _MambaBlock(nn.Module):
    """Simplified Mamba block: depthwise conv + selective gating."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_inner  = d_model * expand
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2)
        self.conv     = nn.Conv1d(self.d_inner, self.d_inner, d_conv, padding=d_conv - 1, groups=self.d_inner)
        self.x_proj   = nn.Linear(self.d_inner, 1 + d_state * 2)
        self.dt_proj  = nn.Linear(1, self.d_inner)
        A_log         = torch.arange(1, d_state + 1, dtype=torch.float32).log()
        self.A_log    = nn.Parameter(A_log.unsqueeze(0).expand(self.d_inner, -1).clone())
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, x):
        B, L, D = x.shape
        h = self.norm(x)
        xz        = self.in_proj(h)
        x_, z     = xz.chunk(2, dim=-1)
        x_        = self.conv(x_.transpose(1, 2))[..., :L].transpose(1, 2)
        x_        = F.silu(x_)
        xd        = self.x_proj(x_)
        dt        = xd[..., :1]
        B_ssm     = xd[..., 1:1 + self.A_log.shape[1]]
        C_ssm     = xd[..., 1 + self.A_log.shape[1]:]
        dt        = F.softplus(self.dt_proj(dt))
        # Parallel selective scan approximation: exponentially weighted conv
        A         = -torch.exp(self.A_log.float())              # [D_inner, N]
        decay     = (dt * A.mean(-1)[None, None, :]).sigmoid()  # gate-like
        y         = x_ * (1 + decay) * F.silu(z)
        return x + self.out_proj(y)

class MambaNO(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, n_modes=16, in_ch=2, **kw):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        self.mambs  = nn.ModuleList([_MambaBlock(hidden_dim) for _ in range(n_layers)])
        self.filt_w = nn.ParameterList([nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5) for _ in range(n_layers)])
        self.n_modes= n_modes
        self.proj   = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Linear(hidden_dim // 2, 1))

    def _spec_filter(self, x, w):
        B, N, H = x.shape
        xf = torch.fft.rfft(x, dim=1)
        m  = min(self.n_modes, xf.shape[1])
        out= torch.einsum('bmc,mco->bmo', xf[:, :m].real, w[:m]) + 0j
        if xf.shape[1] > m:
            out = torch.cat([out, torch.zeros(B, xf.shape[1]-m, H, dtype=out.dtype, device=x.device)], 1)
        return torch.fft.irfft(out, n=N, dim=1)

    def forward(self, u0):
        B = u0.shape[0]
        N = u0.numel() // B
        u0f  = u0.reshape(B, N)
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0f, grid], -1)
        x    = self.lift(x)
        for mamba, w in zip(self.mambs, self.filt_w):
            x = mamba(x)
            x = x + self._spec_filter(x, w)
        return self.proj(x).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# ChebyKAN / cPIKAN_FNO  — Chebyshev KAN Spectral Operator
# ─────────────────────────────────────────────────────────────────────────────
class _ChebyKANLinear(nn.Module):
    def __init__(self, in_f, out_f, degree=5):
        super().__init__()
        self.degree = degree
        s = in_f ** -0.5
        self.base_s = nn.Parameter(torch.ones(out_f, 1))
        self.base_v = nn.Parameter(torch.randn(out_f, in_f) * s)
        self.cheb_w = nn.Parameter(torch.randn(out_f, in_f, degree + 1) * (in_f * (degree + 1)) ** -0.5)

    def forward(self, x):
        sh = x.shape
        x  = x.reshape(-1, sh[-1])
        xt = torch.tanh(x)
        basis = [torch.ones_like(xt)]
        if self.degree > 0: basis.append(xt)
        for _ in range(2, self.degree + 1):
            basis.append(2 * xt * basis[-1] - basis[-2])
        basis = torch.stack(basis, -1)           # [N, in_f, deg+1]
        base  = F.silu(x) @ (self.base_s * self.base_v).t()
        cheb  = torch.einsum('bik,oik->bo', basis, self.cheb_w)
        return (base + cheb).reshape(*sh[:-1], -1)

class cPIKAN_FNO(nn.Module):
    def __init__(self, n_modes, hidden_dim, n_layers, in_ch=2, degree=5, **kw):
        super().__init__()
        self.lift   = nn.Linear(in_ch, hidden_dim)
        # Each block: spectral + chebyshev KAN
        self.wr     = nn.ParameterList([nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5) for _ in range(n_layers)])
        self.wi     = nn.ParameterList([nn.Parameter(torch.randn(n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5) for _ in range(n_layers)])
        self.kans   = nn.ModuleList([_ChebyKANLinear(hidden_dim, hidden_dim, degree) for _ in range(n_layers)])
        self.n_modes= n_modes
        self.n_layers = n_layers
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def _spec(self, x, wr, wi):
        B, N, H = x.shape
        xf = torch.fft.rfft(x, dim=1)
        m  = min(self.n_modes, xf.shape[1])
        xr, xi = xf[:, :m].real, xf[:, :m].imag
        or_ = torch.einsum('bmc,mco->bmo', xr, wr[:m]) - torch.einsum('bmc,mco->bmo', xi, wi[:m])
        oi  = torch.einsum('bmc,mco->bmo', xr, wi[:m]) + torch.einsum('bmc,mco->bmo', xi, wr[:m])
        out = torch.complex(or_, oi)
        if xf.shape[1] > m:
            out = torch.cat([out, torch.zeros(B, xf.shape[1]-m, H, dtype=out.dtype, device=x.device)], 1)
        return torch.fft.irfft(out, n=N, dim=1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for i in range(self.n_layers):
            x = F.gelu(self._spec(x, self.wr[i], self.wi[i]) + self.kans[i](x))
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# GNOT — General Neural Operator Transformer
# ─────────────────────────────────────────────────────────────────────────────
class _GNOTBlock(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        return x + self.ff(self.norm2(x))

class GNOT(nn.Module):
    """GNOT 1-D: transformer over spatial tokens."""
    def __init__(self, hidden_dim, n_layers, n_head=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([_GNOTBlock(hidden_dim, n_head) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(hidden_dim)
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(self.norm(x)))).squeeze(-1)

class _AxialAttn2d(nn.Module):
    def __init__(self, dim, n_heads=4):
        super().__init__()
        self.attn_h = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.attn_w = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ln_h   = nn.LayerNorm(dim)
        self.ln_w   = nn.LayerNorm(dim)

    def forward(self, x):  # [B, H, W, C]
        B, H, W, C = x.shape
        h = self.ln_h(x).permute(0, 2, 1, 3).reshape(B * W, H, C)
        h, _ = self.attn_h(h, h, h)
        x = x + h.reshape(B, W, H, C).permute(0, 2, 1, 3)
        w = self.ln_w(x).reshape(B * H, W, C)
        w, _ = self.attn_w(w, w, w)
        return x + w.reshape(B, H, W, C)

class GNOT2D(nn.Module):
    """GNOT 2-D: axial attention over grid."""
    def __init__(self, hidden_dim, n_layers, n_head=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        self.blocks = nn.ModuleList([_AxialAttn2d(hidden_dim, n_head) for _ in range(n_layers)])
        self.norms  = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.ffs    = nn.ModuleList([nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2), nn.GELU(), nn.Linear(hidden_dim * 2, hidden_dim)) for _ in range(n_layers)])
        self.proj   = nn.Linear(hidden_dim, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for attn, norm, ff in zip(self.blocks, self.norms, self.ffs):
            x = attn(x) + ff(norm(x))
        return self.proj(x).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# Transolver — Physics Attention (1-D and 2-D)
# ─────────────────────────────────────────────────────────────────────────────
class _PhysicsAttn1d(nn.Module):
    def __init__(self, dim, n_head=4, slice_num=32):
        super().__init__()
        assert dim % n_head == 0
        self.n_head   = n_head
        self.slice_num= slice_num
        self.head_dim = dim // n_head
        self.scale    = self.head_dim ** -0.5
        self.to_slice = nn.Linear(dim, n_head * slice_num, bias=False)
        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)
        self.out  = nn.Linear(dim, dim)

    def forward(self, x):  # [B, N, D]
        B, N, D = x.shape
        H, S, d = self.n_head, self.slice_num, self.head_dim
        A = self.to_slice(x).reshape(B, N, H, S).permute(0, 2, 1, 3).softmax(-1)  # [B,H,N,S]
        q = self.to_q(x).reshape(B, N, H, d).permute(0, 2, 1, 3)
        k = self.to_k(x).reshape(B, N, H, d).permute(0, 2, 1, 3)
        v = self.to_v(x).reshape(B, N, H, d).permute(0, 2, 1, 3)
        At = A.transpose(-1, -2)               # [B,H,S,N]
        qs = At @ q; ks = At @ k; vs = At @ v  # [B,H,S,d]
        dots = (qs @ ks.transpose(-1, -2)) * self.scale
        attn = dots.softmax(-1)
        hs   = attn @ vs                        # [B,H,S,d]
        out  = (A @ hs).permute(0, 2, 1, 3).reshape(B, N, D)
        return self.out(out)

class _TransolverBlock1d(nn.Module):
    def __init__(self, dim, n_head=4, slice_num=32):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = _PhysicsAttn1d(dim, n_head, slice_num)
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ff(self.norm2(x))

class Transolver(nn.Module):
    def __init__(self, hidden_dim, n_layers, n_head=4, slice_num=32, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([_TransolverBlock1d(hidden_dim, n_head, slice_num) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)

class Transolver2D(nn.Module):
    def __init__(self, hidden_dim, n_layers, n_head=4, slice_num=32, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        self.blocks = nn.ModuleList([_TransolverBlock1d(hidden_dim, n_head, slice_num) for _ in range(n_layers)])
        self.proj   = nn.Linear(hidden_dim, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x).reshape(B, N1 * N2, -1)
        for blk in self.blocks: x = blk(x)
        return self.proj(x).squeeze(-1).reshape(B, N1, N2)


# ─────────────────────────────────────────────────────────────────────────────
# 2-D Variants: SNO2D, FEDONet2D, HANO2D, VSMNO2D
# ─────────────────────────────────────────────────────────────────────────────
class SNO2D(nn.Module):
    """Spectral Neural Operator — pure spectral layers (2-D)."""
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift = nn.Linear(3, hidden_dim)
        self.wr   = nn.ParameterList([nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5) for _ in range(n_layers)])
        self.wi   = nn.ParameterList([nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5) for _ in range(n_layers)])
        self.n_modes  = n_modes
        self.n_layers = n_layers
        self.proj = nn.Linear(hidden_dim, 1)

    def _spec2d(self, x, wr, wi):
        B, N1, N2, H = x.shape
        xf = torch.fft.rfft2(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        m  = self.n_modes
        xr, xi = xf[:, :m, :m].real, xf[:, :m, :m].imag
        or_ = torch.einsum('bmki,mkio->bmko', xr, wr) - torch.einsum('bmki,mkio->bmko', xi, wi)
        oi  = torch.einsum('bmki,mkio->bmko', xr, wi) + torch.einsum('bmki,mkio->bmko', xi, wr)
        out = torch.zeros_like(xf)
        out[:, :m, :m] = torch.complex(or_, oi)
        return torch.fft.irfft2(out.permute(0, 3, 1, 2), s=(N1, N2)).permute(0, 2, 3, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for i in range(self.n_layers):
            x = F.gelu(self._spec2d(x, self.wr[i], self.wi[i]))
        return self.proj(x).squeeze(-1)

class FEDONet2D(nn.Module):
    """Fourier-Enhanced DeepONet (branch=FNO+MLP, trunk=coordinate MLP)."""
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift         = nn.Linear(3, hidden_dim)
        self.branch_spec  = SNO2D.__dict__['_spec2d']   # reuse via composition
        self.wr = nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.wi = nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)
        self.n_modes = n_modes
        self.branch_mlp = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.trunk      = nn.Sequential(nn.Linear(2, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.proj       = nn.Linear(hidden_dim, 1)

    def _spec(self, x):
        B, N1, N2, H = x.shape
        xf = torch.fft.rfft2(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        m  = self.n_modes
        xr, xi = xf[:, :m, :m].real, xf[:, :m, :m].imag
        or_ = torch.einsum('bmki,mkio->bmko', xr, self.wr) - torch.einsum('bmki,mkio->bmko', xi, self.wi)
        oi  = torch.einsum('bmki,mkio->bmko', xr, self.wi) + torch.einsum('bmki,mkio->bmko', xi, self.wr)
        out = torch.zeros_like(xf); out[:, :m, :m] = torch.complex(or_, oi)
        return torch.fft.irfft2(out.permute(0, 3, 1, 2), s=(N1, N2)).permute(0, 2, 3, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        b_out = F.gelu(self._spec(x) + self.branch_mlp(x))
        g1, g2 = _grid2d(B, N1, N2, u0.device)
        coords = torch.stack([g1, g2], -1)  # [B, N1, N2, 2]
        t_out  = self.trunk(coords)
        return self.proj(b_out * t_out).squeeze(-1)

class HANO2D(nn.Module):
    """Hierarchical Attention Neural Operator: FNO + axial attention."""
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        self.specs  = nn.ModuleList()
        self.attns  = nn.ModuleList()
        self.wls    = nn.ModuleList()
        self.n_modes= n_modes
        for _ in range(n_layers):
            self.specs.append(nn.ParameterDict({'wr': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5),
                                                 'wi': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)}))
            self.attns.append(_AxialAttn2d(hidden_dim, n_heads=4))
            self.wls.append(nn.Linear(hidden_dim, hidden_dim))
        self.proj = nn.Linear(hidden_dim, 1)

    def _spec(self, x, wr, wi):
        B, N1, N2, H = x.shape
        m  = self.n_modes
        xf = torch.fft.rfft2(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        xr, xi = xf[:, :m, :m].real, xf[:, :m, :m].imag
        or_ = torch.einsum('bmki,mkio->bmko', xr, wr) - torch.einsum('bmki,mkio->bmko', xi, wi)
        oi  = torch.einsum('bmki,mkio->bmko', xr, wi) + torch.einsum('bmki,mkio->bmko', xi, wr)
        out = torch.zeros_like(xf); out[:, :m, :m] = torch.complex(or_, oi)
        return torch.fft.irfft2(out.permute(0, 3, 1, 2), s=(N1, N2)).permute(0, 2, 3, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for sp, at, wl in zip(self.specs, self.attns, self.wls):
            x = F.gelu(self._spec(x, sp['wr'], sp['wi']) + at(x) + wl(x))
        return self.proj(x).squeeze(-1)

class VSMNO2D(nn.Module):
    """Variational Spectral Mixture Neural Operator (multi-resolution 2-D)."""
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        m2 = max(2, n_modes // 2)
        self.specs_hi = nn.ModuleList([nn.ParameterDict({'wr': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'wi': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5)}) for _ in range(n_layers)])
        self.specs_lo = nn.ModuleList([nn.ParameterDict({'wr': nn.Parameter(torch.randn(m2, m2, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'wi': nn.Parameter(torch.randn(m2, m2, hidden_dim, hidden_dim) * hidden_dim ** -0.5)}) for _ in range(n_layers)])
        self.wls = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)])
        self.w_mix = nn.Parameter(torch.ones(2))
        self.n_modes = n_modes; self.m2 = m2
        self.proj = nn.Linear(hidden_dim, 1)

    def _spec(self, x, sp, m):
        B, N1, N2, H = x.shape
        xf = torch.fft.rfft2(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        xr, xi = xf[:, :m, :m].real, xf[:, :m, :m].imag
        or_ = torch.einsum('bmki,mkio->bmko', xr, sp['wr'][:m, :m]) - torch.einsum('bmki,mkio->bmko', xi, sp['wi'][:m, :m])
        oi  = torch.einsum('bmki,mkio->bmko', xr, sp['wi'][:m, :m]) + torch.einsum('bmki,mkio->bmko', xi, sp['wr'][:m, :m])
        out = torch.zeros_like(xf); out[:, :m, :m] = torch.complex(or_, oi)
        return torch.fft.irfft2(out.permute(0, 3, 1, 2), s=(N1, N2)).permute(0, 2, 3, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        w = self.w_mix.softmax(0)
        for sh, sl, wl in zip(self.specs_hi, self.specs_lo, self.wls):
            x = F.gelu(w[0] * self._spec(x, sh, self.n_modes) + w[1] * self._spec(x, sl, self.m2) + wl(x))
        return self.proj(x).squeeze(-1)


# ───────────────────────────────────────────────────────────────────────���─────
# Hybrid DeepONet variants (2-D)
# ─────────────────────────────────────────────────────────────────────────────
def _fno_block2d_pt(wr, wi, n_modes, x):
    """Single FNO2D spectral+linear pass (functional helper)."""
    B, N1, N2, H = x.shape
    m  = n_modes
    xf = torch.fft.rfft2(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
    xr, xi = xf[:, :m, :m].real, xf[:, :m, :m].imag
    or_ = torch.einsum('bmki,mkio->bmko', xr, wr) - torch.einsum('bmki,mkio->bmko', xi, wi)
    oi  = torch.einsum('bmki,mkio->bmko', xr, wi) + torch.einsum('bmki,mkio->bmko', xi, wr)
    out = torch.zeros_like(xf); out[:, :m, :m] = torch.complex(or_, oi)
    return torch.fft.irfft2(out.permute(0, 3, 1, 2), s=(N1, N2)).permute(0, 2, 3, 1)

class HybridFNODeepONet2D(nn.Module):
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        self.specs  = nn.ModuleList([nn.ParameterDict({'wr': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'wi': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'w': nn.Linear(hidden_dim, hidden_dim)}) for _ in range(n_layers)])
        self.n_modes= n_modes
        self.trunk  = nn.Sequential(nn.Linear(2, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.proj   = nn.Linear(hidden_dim, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for sp in self.specs:
            x = F.gelu(_fno_block2d_pt(sp['wr'], sp['wi'], self.n_modes, x) + sp['w'](x))
        g1, g2 = _grid2d(B, N1, N2, u0.device)
        t = self.trunk(torch.stack([g1, g2], -1))
        return self.proj(x * t).squeeze(-1)

class HybridDecoderDeepONet2D(nn.Module):
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(3, hidden_dim)
        self.specs  = nn.ModuleList([nn.ParameterDict({'wr': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'wi': nn.Parameter(torch.randn(n_modes, n_modes, hidden_dim, hidden_dim) * hidden_dim ** -0.5), 'w': nn.Linear(hidden_dim, hidden_dim)}) for _ in range(n_layers)])
        self.n_modes= n_modes
        self.trunk  = nn.Sequential(nn.Linear(2, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))
        self.proj   = nn.Linear(hidden_dim, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for sp in self.specs:
            x = F.gelu(_fno_block2d_pt(sp['wr'], sp['wi'], self.n_modes, x) + sp['w'](x))
        g1, g2 = _grid2d(B, N1, N2, u0.device)
        t = self.trunk(torch.stack([g1, g2], -1))
        return self.proj(x * t).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# WNO — Wavelet Neural Operator (1-D Haar)
# ─────────────────────────────────────────────────────────────────────────────
def _haar_fwd(x, levels):
    sq2 = math.sqrt(2); details = []; a = x
    for _ in range(levels):
        B, N, C = a.shape; p = a.reshape(B, N // 2, 2, C)
        lo = (p[:, :, 0] + p[:, :, 1]) / sq2
        hi = (p[:, :, 0] - p[:, :, 1]) / sq2
        details.append(hi); a = lo
    return details, a

def _haar_inv(details, a):
    sq2 = math.sqrt(2)
    for hi in reversed(details):
        lo = a; B, N, C = lo.shape
        out = torch.zeros(B, N * 2, C, device=lo.device, dtype=lo.dtype)
        out[:, ::2]  = (lo + hi) / sq2
        out[:, 1::2] = (lo - hi) / sq2
        a = out
    return a

class _WNOBlock1d(nn.Module):
    def __init__(self, channels, n_levels=3, n_modes=16):
        super().__init__()
        self.n_levels = n_levels
        scale = channels ** -0.5
        # learnable weights per level
        self.Wt = nn.ParameterList([nn.Parameter(torch.randn(channels, channels) * scale) for _ in range(n_levels)])
        self.Wa = nn.Parameter(torch.randn(channels, channels) * scale)
        self.W  = nn.Linear(channels, channels)

    def forward(self, x):
        details, a = _haar_fwd(x, self.n_levels)
        a_out = torch.einsum('bnc,cd->bnd', a, self.Wa)
        d_out = [torch.einsum('bnc,cd->bnd', d, self.Wt[i]) for i, d in enumerate(details)]
        return F.gelu(x + _haar_inv(d_out, a_out) + self.W(x))

class WNO(nn.Module):
    def __init__(self, n_modes=16, hidden_dim=64, n_layers=4, n_levels=3, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([_WNOBlock1d(hidden_dim, n_levels) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# AFNO — Adaptive Fourier Neural Operator
# ─────────────────────────────────────────────────────────────────────────────
class _AFNOBlock1d(nn.Module):
    def __init__(self, hidden_dim, softshrink=0.01):
        super().__init__()
        self.norm     = nn.LayerNorm(hidden_dim)
        self.mlp      = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim * 2), nn.GELU(), nn.Linear(hidden_dim * 2, hidden_dim * 2))
        self.softshrink = softshrink
        self.W        = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        B, N, H = x.shape
        h  = self.norm(x)
        xf = torch.fft.rfft(h, dim=1)                    # [B, M, H] complex
        xc = torch.cat([xf.real, xf.imag], -1)           # [B, M, 2H]
        xc = self.mlp(xc)
        xc = F.softshrink(xc, self.softshrink)
        xf_out = torch.complex(xc[..., :H], xc[..., H:])
        out = torch.fft.irfft(xf_out, n=N, dim=1)
        return x + out + self.W(h)

class AFNO(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([_AFNOBlock1d(hidden_dim) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# TFNO — Tucker-Factorized FNO (1-D)
# ─────────────────────────────────────────────────────────────────────────────
class _TuckerSpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, n_modes, rank_ratio=0.5):
        super().__init__()
        r_m = max(1, int(n_modes * rank_ratio))
        r_i = max(1, int(in_ch  * rank_ratio))
        r_o = max(1, int(out_ch * rank_ratio))
        s   = (in_ch * out_ch) ** -0.5
        self.Gr = nn.Parameter(torch.randn(r_m, r_i, r_o) * s)
        self.Gi = nn.Parameter(torch.randn(r_m, r_i, r_o) * s)
        self.Um = nn.Parameter(torch.randn(n_modes, r_m) * s)
        self.Ui = nn.Parameter(torch.randn(in_ch,   r_i) * s)
        self.Uo = nn.Parameter(torch.randn(out_ch,  r_o) * s)
        self.n_modes = n_modes; self.out_ch = out_ch

    def forward(self, x):
        B, N, _ = x.shape
        xf = torch.fft.rfft(x, dim=1)
        xr, xi = xf[:, :self.n_modes].real, xf[:, :self.n_modes].imag
        Wr = torch.einsum('abc,ma,ib,oc->mio', self.Gr, self.Um, self.Ui, self.Uo)
        Wi = torch.einsum('abc,ma,ib,oc->mio', self.Gi, self.Um, self.Ui, self.Uo)
        or_ = torch.einsum('bmi,mio->bmo', xr, Wr) - torch.einsum('bmi,mio->bmo', xi, Wi)
        oi  = torch.einsum('bmi,mio->bmo', xr, Wi) + torch.einsum('bmi,mio->bmo', xi, Wr)
        out = torch.complex(or_, oi)
        n_rfft = N // 2 + 1
        if n_rfft > self.n_modes:
            out = torch.cat([out, torch.zeros(B, n_rfft - self.n_modes, self.out_ch, dtype=out.dtype, device=x.device)], 1)
        return torch.fft.irfft(out, n=N, dim=1)

class TFNO(nn.Module):
    """Tucker-Factorized FNO (1-D)."""
    def __init__(self, n_modes=16, hidden_dim=64, n_layers=4, **kw):
        super().__init__()
        self.lift   = nn.Linear(2, hidden_dim)
        self.specs  = nn.ModuleList([_TuckerSpectralConv1d(hidden_dim, hidden_dim, n_modes) for _ in range(n_layers)])
        self.ws     = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for sp, w in zip(self.specs, self.ws):
            x = F.gelu(sp(x) + w(x))
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)

class TFNO2D(nn.Module):
    """Tucker-Factorized FNO (2-D)."""
    def __init__(self, n_modes=8, hidden_dim=32, n_layers=4, **kw):
        super().__init__()
        from models.fno import FNOBlock2d  # reuse existing PyTorch block
        self.lift   = nn.Linear(3, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock2d(hidden_dim, n_modes, n_modes) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        x, B, N1, N2 = _lift2d(u0)
        x = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)


# ─────────────────────────────────────────────────────────────────────────────
# Hamiltonian Neural Operator & EnergyFNO
# ─────────────────────────────────────────────────────────────────────────────
class HamiltonianNO(nn.Module):
    """Hamiltonian Neural Operator: encode IC → (q,p), compute ∂H/∂p as output."""
    def __init__(self, hidden_dim=64, n_layers=4, n_modes=16, **kw):
        super().__init__()
        N = 64  # fixed grid size matching benchmark
        self.enc_q   = nn.Sequential(nn.Linear(N, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, hidden_dim))
        self.enc_p   = nn.Sequential(nn.Linear(N, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, hidden_dim))
        layers = [nn.Linear(hidden_dim * 2, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.H_net   = nn.Sequential(*layers)
        self.alpha   = nn.Parameter(torch.tensor(0.1))
        # Decode latent ∂H/∂p back to grid
        self.decode  = nn.Linear(hidden_dim, N)

    def forward(self, u0):
        B, N = u0.shape
        q    = self.enc_q(u0)
        p    = self.enc_p(u0)
        qp   = torch.cat([q, p], -1)
        # ∂H/∂p via autograd
        p_param = p.detach().requires_grad_(True)
        qp_ag   = torch.cat([q.detach(), p_param], -1)
        H       = self.H_net(qp_ag).sum()
        dHdp    = torch.autograd.grad(H, p_param, create_graph=self.training)[0]
        return u0 + self.alpha * self.decode(dHdp)

class EnergyFNO(nn.Module):
    """FNO with soft energy conservation auxiliary loss (EnergyConservingFNO)."""
    def __init__(self, n_modes=16, hidden_dim=64, n_layers=4, **kw):
        super().__init__()
        from models.fno import FNOBlock1d  # reuse existing PyTorch FNO block
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock1d(hidden_dim, n_modes) for _ in range(n_layers)])
        self.proj1  = nn.Linear(hidden_dim, hidden_dim // 2)
        self.proj2  = nn.Linear(hidden_dim // 2, 1)

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        x    = torch.stack([u0, grid], -1)
        x    = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj2(F.gelu(self.proj1(x))).squeeze(-1)

# Alias used in experiments.yaml
HNN = HamiltonianNO


# ─────────────────────────────────────────────────────────────────────────────
# NeuralODE — Neural ODE with Euler/RK4 integration
# ─────────────────────────────────────────────────────────────────────────────
class _DerivNet1d(nn.Module):
    def __init__(self, hidden_dim=32, n_modes=16, n_layers=3):
        super().__init__()
        from models.fno import FNOBlock1d
        self.lift   = nn.Linear(2, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock1d(hidden_dim, n_modes) for _ in range(n_layers)])
        self.proj   = nn.Linear(hidden_dim, 1)

    def forward(self, u, grid):
        x = torch.stack([u, grid], -1)
        x = self.lift(x)
        for blk in self.blocks: x = blk(x)
        return self.proj(x).squeeze(-1)

class NeuralODE(nn.Module):
    """Neural ODE for 1-D PDEs: integrates du/dt = F_θ(u, x)."""
    def __init__(self, n_modes=16, hidden_dim=64, n_layers=4, n_steps=20, **kw):
        super().__init__()
        self.deriv   = _DerivNet1d(hidden_dim, n_modes, max(2, n_layers - 1))
        self.n_steps = n_steps
        self.T       = 1.0

    def forward(self, u0):
        B, N = u0.shape
        grid = _grid1d(B, N, u0.device)
        u    = u0
        dt   = self.T / self.n_steps
        for _ in range(self.n_steps):
            # RK4
            k1 = self.deriv(u, grid)
            k2 = self.deriv(u + 0.5 * dt * k1, grid)
            k3 = self.deriv(u + 0.5 * dt * k2, grid)
            k4 = self.deriv(u + dt * k3, grid)
            u  = u + (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        return u


# ─────────────────────────────────────────────────────────────────────────────
# TimeDeepONet — Time-aware two-branch DeepONet
# ─────────────────────────────────────────────────────────────────────────────
class TimeDeepONet(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, n_modes=16, **kw):
        super().__init__()
        N = 64  # fixed grid

        def _mlp(dims):
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i + 1]))
                if i < len(dims) - 2:
                    layers += [nn.GELU(), nn.LayerNorm(dims[i + 1])]
            return nn.Sequential(*layers)

        p = hidden_dim
        self.branch1 = _mlp([N] + [hidden_dim] * n_layers + [p])
        self.branch2 = _mlp([1] + [hidden_dim] * n_layers + [p])
        self.trunk   = _mlp([1] + [hidden_dim] * n_layers + [p])
        self.bias    = nn.Parameter(torch.zeros(1))

    def forward(self, u0):
        B, N = u0.shape
        b1   = self.branch1(u0)                                          # [B, p]
        t_val= torch.ones(B, 1, device=u0.device)                       # t=1 (final state)
        b2   = self.branch2(t_val)                                       # [B, p]
        x_q  = torch.linspace(0, 1, N, device=u0.device).unsqueeze(-1)  # [N, 1]
        t_out= self.trunk(x_q)                                           # [N, p]
        return torch.einsum('bp,np->bn', b1 * b2, t_out) + self.bias


# ─────────────────────────────────────────────────────────────────────────────
# DeepONet — standard DeepONet (1-D)
# ─────────────────────────────────────────────────────────────────────────────
class DeepONet(nn.Module):
    def __init__(self, hidden_dim=64, n_layers=4, **kw):
        super().__init__()
        N = 64

        def _net(in_d):
            layers, d = [], in_d
            for _ in range(n_layers):
                layers += [nn.Linear(d, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)]
                d = hidden_dim
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            return nn.Sequential(*layers)

        self.branch = _net(N)
        self.trunk  = _net(1)
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, u0):
        B, N = u0.shape
        b    = self.branch(u0)                                           # [B, hidden]
        x_q  = torch.linspace(0, 1, N, device=u0.device).unsqueeze(-1)  # [N, 1]
        t    = self.trunk(x_q)                                           # [N, hidden]
        return torch.einsum('bh,nh->bn', b, t) + self.bias


# ─────────────────────────────────────────────────────────────────────────────
# PACMANN & PINN (simple FNO / coordinate-MLP baselines)
# ─────────────────────────────────────────────────────────────────────────────
class PACMANN(nn.Module):
    """PACMANN stub (FNO1D with residual connection)."""
    def __init__(self, n_modes=16, hidden_dim=64, n_layers=4, **kw):
        super().__init__()
        from models.fno import FNOBlock1d
        self.lift   = nn.Linear(1, hidden_dim)
        self.blocks = nn.ModuleList([FNOBlock1d(hidden_dim, n_modes) for _ in range(n_layers)])
        self.proj   = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, u0):
        B, N = u0.shape
        x = self.lift(u0.unsqueeze(-1))
        for blk in self.blocks: x = x + blk(x)
        return self.proj(x).squeeze(-1)

class PINN(nn.Module):
    """Coordinate-based PINN surrogate for operator learning."""
    def __init__(self, hidden_dim=128, n_layers=4, **kw):
        super().__init__()
        N = 64
        layers = [nn.Linear(N + 1, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim)]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, u0):
        B, N = u0.shape
        x_q  = torch.linspace(0, 1, N, device=u0.device).unsqueeze(0).expand(B, -1)
        inp  = torch.cat([u0, x_q[:, :1]], -1)  # IC + one coord
        # Actually we want [B, N] output — use IC + grid per point
        u0_exp = u0.unsqueeze(1).expand(-1, N, -1)  # [B, N, N]
        g      = x_q.unsqueeze(-1)                   # [B, N, 1]
        inp    = torch.cat([u0_exp, g], -1)           # [B, N, N+1]
        return self.net(inp).squeeze(-1)


print('Extended model registry ready:')
print('  1-D: SSNO, MemNO, MambaNO, cPIKAN_FNO, GNOT, Transolver, WNO, AFNO, TFNO')
print('       HNN/HamiltonianNO, EnergyFNO, NeuralODE, TimeDeepONet, DeepONet, PACMANN, PINN')
print('  2-D: SNO2D, FEDONet2D, HANO2D, VSMNO2D, HybridFNODeepONet2D,')
print('       HybridDecoderDeepONet2D, GNOT2D, Transolver2D, TFNO2D')

## Cell 7 — Loss functions

In [ ]:
def relative_l2(pred, y):
    axes = tuple(range(1, y.ndim))
    diff = pred - y
    num  = torch.sqrt((diff ** 2).mean(dim=axes))
    den  = torch.sqrt((y ** 2).mean(dim=axes)) + 1e-8
    return (num / den).mean()

def h1_loss(pred, y, alpha=0.1):
    base = relative_l2(pred, y)
    # spectral derivative via rfft (1D only)
    if pred.ndim == 2:
        N  = pred.shape[1]
        k  = torch.fft.rfftfreq(N, d=1.0/N).to(pred.device)
        dp = torch.fft.irfft(1j * k * torch.fft.rfft(pred, dim=1), n=N, dim=1)
        dy = torch.fft.irfft(1j * k * torch.fft.rfft(y,    dim=1), n=N, dim=1)
        base = base + alpha * relative_l2(dp, dy)
    return base

def spectral_loss(pred, y, high_freq_weight=2.0):
    if pred.ndim != 2:
        return relative_l2(pred, y)
    N     = pred.shape[1]
    k     = torch.fft.rfftfreq(N, d=1.0/N).to(pred.device) * N
    w     = 1.0 + (k / (N // 2)) ** high_freq_weight
    pf    = torch.fft.rfft(pred, dim=1)
    yf    = torch.fft.rfft(y,    dim=1)
    diff  = (pf - yf).abs() * w[None, :]
    ynorm = yf.abs() * w[None, :]
    return (diff.mean(dim=1) / (ynorm.mean(dim=1) + 1e-8)).mean()

def relative_l1(pred, y):
    axes = tuple(range(1, y.ndim))
    num  = (pred - y).abs().mean(dim=axes)
    den  = y.abs().mean(dim=axes) + 1e-8
    return (num / den).mean()

LOSS_REGISTRY = {
    'l2_rel':    relative_l2,
    'h1':        h1_loss,
    'h1_strong': lambda p, y: h1_loss(p, y, alpha=1.0),
    'spectral':  spectral_loss,
    'l1_rel':    relative_l1,
    'mse':       nn.MSELoss(),
}

def get_loss_fn(name):
    return LOSS_REGISTRY[name]

print('Loss functions ready:', list(LOSS_REGISTRY))

## Cell 8 — Training harness

In [ ]:
import copy

WARMUP_RATIO   = 0.1
WARMDOWN_RATIO = 0.2
FINAL_LR_FRAC  = 0.01

def lr_schedule(progress, warmup=WARMUP_RATIO, wardown=WARMDOWN_RATIO, final=FINAL_LR_FRAC):
    if progress < warmup:
        return progress / warmup
    if progress < 1.0 - wardown:
        return 1.0
    t = (1.0 - progress) / wardown
    return t + (1.0 - t) * final


def evaluate_l2(model, benchmark, device, batch_size=EVAL_BATCH):
    """Compute relative L2 on the fixed validation set."""
    model.eval()
    val_gen = make_torch_loader(benchmark, 'val', batch_size, device)
    n_batches = math.ceil(N_VAL / batch_size)
    total_num, total_den = 0.0, 0.0
    with torch.no_grad():
        for _ in range(n_batches):
            x, y = next(val_gen)
            pred = model(x)
            axes = tuple(range(1, y.ndim))
            num  = torch.sqrt(((pred - y) ** 2).mean(dim=axes))
            den  = torch.sqrt((y ** 2).mean(dim=axes)) + 1e-8
            total_num += num.sum().item()
            total_den += den.sum().item()
    model.train()
    return total_num / max(total_den, 1e-8)


def train_experiment(cfg: ExperimentConfig, device=DEVICE, log_interval=50, verbose=True):
    """
    Train one experiment config.  Returns dict with all metrics.
    Writes a log file to LOGS_DIR/<cfg.name>.log
    """
    log_path  = LOGS_DIR / f'{cfg.name}.log'
    log_lines = []

    def log(msg):
        log_lines.append(msg)
        if verbose:
            print(msg)

    log(f'=== {cfg.name} ===')
    log(f'benchmark={cfg.benchmark}  model={cfg.model}')
    log(f'hidden={cfg.hidden_dim}  layers={cfg.n_layers}  modes={cfg.n_modes}')
    log(f'budget={cfg.budget_s}s  loss={cfg.loss_type}  device={device}')

    # Build model
    model     = build_model(cfg, device)
    n_params  = count_params(model)
    log(f'params: {n_params / 1e6:.3f}M')

    # Optimizer + LR scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
    loss_fn   = get_loss_fn(cfg.loss_type)

    # Data
    train_gen = make_torch_loader(cfg.benchmark, 'train', cfg.batch_size, device)

    # Training state
    best_val      = float('inf')
    best_state    = None
    loss_history  = []
    step          = 0
    t_start       = time.time()
    last_val_time = t_start
    peak_vram_mb  = 0.0
    VAL_INTERVAL  = max(30, cfg.budget_s // 10)  # validate ~10 times

    model.train()
    log('Training...')

    # Telemetry file (mirrors .vram_telemetry_<name> for dashboard)
    telem_path = REPO_PATH / f'.vram_telemetry_{cfg.name}'

    while True:
        elapsed  = time.time() - t_start
        progress = min(elapsed / cfg.budget_s, 1.0)
        if elapsed >= cfg.budget_s:
            break

        # LR update
        scale = lr_schedule(progress)
        for pg in optimizer.param_groups:
            pg['lr'] = cfg.lr * scale

        x, y = next(train_gen)

        # Optional spatial-shift augmentation (periodic BCs)
        if cfg.augment and x.ndim == 2:
            shift = torch.randint(0, x.shape[1], (1,)).item()
            x = torch.roll(x, shift, dims=1)
            y = torch.roll(y, shift, dims=1)

        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)

        # NaN/Inf detection
        if torch.isnan(loss) or torch.isinf(loss):
            log('!!! DETECTED NAN/INF LOSS — ABORTING !!!')
            return {
                'val_l2_rel': 1.0, 'status': 'crash_nan', 'training_s': time.time() - t_start,
                'peak_vram_mb': peak_vram_mb, 'num_steps': step, 'n_params_M': n_params / 1e6,
                'model': model, 'loss_history': loss_history
            }
        loss.backward()

        if cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        optimizer.step()
        step += 1

        loss_val = loss.item()
        loss_history.append(loss_val)

        # VRAM tracking
        if device == 'cuda':
            mem_mb = torch.cuda.memory_allocated() / 1e6
            peak_vram_mb = max(peak_vram_mb, mem_mb)

        if step % log_interval == 0:
            log(f'  step={step:5d}  loss={loss_val:.6f}  lr={optimizer.param_groups[0]["lr"]:.2e}'
                f'  elapsed={elapsed:.0f}s')

        # Write telemetry for dashboard
        if step % 20 == 0:
            recent = loss_history[-50:]
            telem  = {
                'step': step, 'loss': loss_val, 'loss_history': recent,
                'elapsed': elapsed, 'progress': progress,
                'remaining_s': max(0, cfg.budget_s - elapsed),
                'peak_vram_mb': peak_vram_mb, 'device': device,
            }
            telem_path.write_text(json.dumps(telem))

        # Periodic validation
        if time.time() - last_val_time >= VAL_INTERVAL:
            val_err = evaluate_l2(model, cfg.benchmark, device)
            log(f'  [VAL] step={step}  val_l2_rel={val_err:.6f}  elapsed={elapsed:.0f}s')
            if val_err < best_val:
                best_val   = val_err
                best_state = copy.deepcopy(model.state_dict())
            last_val_time = time.time()

    # Final evaluation with best weights
    if best_state is not None:
        model.load_state_dict(best_state)
    val_l2_rel = evaluate_l2(model, cfg.benchmark, device)
    train_time = time.time() - t_start

    # Final CUDA peak
    if device == 'cuda':
        peak_vram_mb = torch.cuda.max_memory_allocated() / 1e6
        torch.cuda.reset_peak_memory_stats()

    log(f'--- DONE ---')
    log(f'val_l2_rel: {val_l2_rel:.6f}')
    log(f'training_seconds: {train_time:.1f}')
    log(f'peak_vram_mb: {peak_vram_mb:.1f}')
    log(f'num_steps: {step}')
    log(f'num_params_M: {n_params/1e6:.3f}')

    # Write log file
    log_path.write_text('\n'.join(log_lines))

    # Clean up telemetry
    if telem_path.exists():
        telem_path.unlink()

    return {
        'val_l2_rel':   val_l2_rel,
        'training_s':   train_time,
        'peak_vram_mb': peak_vram_mb,
        'num_steps':    step,
        'n_params_M':   n_params / 1e6,
        'model':        model,
        'loss_history': loss_history,
    }

print('Training harness ready')

## Cell 9 — Results logger (writes results.json + results.tsv)

In [ ]:
import subprocess as _sp
from dataclasses import asdict

def _git_commit():
    try:
        r = _sp.run(['git', '-C', str(REPO_PATH), 'rev-parse', '--short', 'HEAD'],
                    capture_output=True, text=True)
        return r.stdout.strip() or 'colab'
    except Exception:
        return 'colab'


def _load_results():
    if RESULTS_JSON.exists():
        return json.loads(RESULTS_JSON.read_text())
    return []


def _save_results(experiments):
    RESULTS_JSON.write_text(json.dumps(experiments, indent=2))
    # Sync TSV
    header = ['commit', 'benchmark', 'model', 'val_l2_rel', 'memory_gb', 'status', 'description']
    rows   = []
    for e in experiments:
        rows.append([
            str(e.get('commit', 'colab')),
            str(e['benchmark']),
            str(e['model']),
            f"{e['val_l2_rel']:.6f}",
            f"{e.get('memory_gb', 0):.2f}",
            str(e.get('status', 'keep')),
            str(e.get('description', '')),
        ])
    with open(RESULTS_TSV, 'w', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(header)
        w.writerows(rows)


def log_result(cfg: ExperimentConfig, metrics: dict, status='keep',
               conclusion='', parent_name=''):
    """
    Write experiment result to results.json (and sync results.tsv).
    Compatible with the MLX dashboard format.
    """
    experiments = _load_results()

    # Resolve parent_id
    pname = parent_name or cfg.parent_name
    parent_id = None
    if pname:
        for e in experiments:
            if e.get('config', {}).get('name') == pname:
                parent_id = e['id']
                break

    exp_id = f"{cfg.benchmark}_{cfg.model}_{int(time.time())}_{uuid.uuid4().hex[:6]}"

    node = {
        'id':          exp_id,
        'parent_id':   parent_id,
        'timestamp':   int(time.time()),
        'benchmark':   cfg.benchmark,
        'model':       cfg.model,
        'val_l2_rel':  metrics['val_l2_rel'],
        'memory_gb':   metrics.get('peak_vram_mb', 0) / 1024,
        'status':      status,
        'description': (
            f"{cfg.model} h={cfg.hidden_dim} l={cfg.n_layers} m={cfg.n_modes} "
            f"[colab/{DEVICE}] val={metrics['val_l2_rel']:.4f}"
        ),
        'commit':      _git_commit(),
        'config':      asdict(cfg),
        'rationale':   cfg.rationale,
        'conclusion':  conclusion,
        'diag': {
            'training_seconds': metrics.get('training_s', 0),
            'num_steps':        metrics.get('num_steps', 0),
            'n_params_M':       metrics.get('n_params_M', 0),
            'device':           DEVICE,
            'framework':        'pytorch',
            'low_freq_error':   metrics.get('spec_bias', {}).get('low_freq_error'),
            'high_freq_error':  metrics.get('spec_bias', {}).get('high_freq_error'),
        },
    }

    experiments.append(node)
    _save_results(experiments)
    print(f'Logged: {exp_id}')
    print(f'  val_l2_rel = {metrics["val_l2_rel"]:.6f}  status={status}')
    return exp_id


def is_done(cfg_name):
    """Return True if this experiment name already exists in results.json."""
    for e in _load_results():
        if e.get('config', {}).get('name') == cfg_name:
            return True
    return False


print('Results logger ready')

## Cell 9.5 — Advanced Diagnostics (Spectral Bias & Inspector PNGs)

In [ ]:
def calculate_spectral_bias(pred, truth):
    """Calculate error magnitude across Fourier modes (NumPy)."""
    p, t = np.array(pred), np.array(truth)
    if p.ndim == 3 and p.shape[-1] == 1: p = p.squeeze(-1)
    if t.ndim == 3 and t.shape[-1] == 1: t = t.squeeze(-1)
    if p.ndim == 1: p, t = p[None, :], t[None, :]

    p_ft = np.abs(np.fft.rfft(p, axis=-1))
    t_ft = np.abs(np.fft.rfft(truth, axis=-1))
    err_ft = np.abs(p_ft - t_ft).mean(axis=0)
    total_energy = t_ft.mean(axis=0).sum() + 1e-8

    n_modes = len(err_ft)
    low_cutoff = n_modes // 4
    high_cutoff = n_modes // 2

    return {
        "low_freq_error":  float(err_ft[:low_cutoff].sum() / total_energy),
        "mid_freq_error":  float(err_ft[low_cutoff:high_cutoff].sum() / total_energy),
        "high_freq_error": float(err_ft[high_cutoff:].sum() / total_energy),
        "spectral_gap":    float(np.max(err_ft))
    }

def generate_inspect_png(exp_id, inputs, truth, pred, benchmark):
    """Generate side-by-side PNG for the Dashboard Inspector."""
    is_1d = truth.ndim == 2
    fig_path = FIGS_DIR / f'inspect_{exp_id}.png'
    
    if is_1d:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        x = np.linspace(0, 1, truth.shape[1])
        axes[0].plot(x, truth[0], 'k-', label='Truth')
        axes[0].plot(x, pred[0], 'r--', label='Pred')
        axes[0].set_title("Solution Comparison")
        axes[0].legend()
        axes[1].plot(x, np.abs(truth[0] - pred[0]), 'crimson')
        axes[1].set_title("Spatial Error |y - y^|")
        p_ft = np.abs(np.fft.rfft(pred[0]))
        t_ft = np.abs(np.fft.rfft(truth[0]))
        axes[2].bar(range(len(p_ft)), np.abs(p_ft - t_ft), color='indigo')
        axes[2].set_title("Spectral Error Magnitude")
        axes[2].set_yscale('log')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(truth[0], cmap='viridis'); axes[0].set_title("Ground Truth")
        axes[1].imshow(pred[0], cmap='viridis'); axes[1].set_title("Prediction")
        axes[2].imshow(np.abs(truth[0] - pred[0]), cmap='inferno'); axes[2].set_title("Absolute Error")
    
    plt.tight_layout()
    fig.savefig(fig_path, dpi=100)
    plt.close(fig)
    return str(fig_path)

print('Diagnostics ready')

## Cell 10 — Run a single experiment

In [ ]:
import matplotlib.pyplot as plt

def run_and_log(cfg: ExperimentConfig, skip_if_done=True):
    """Train, log, and plot one experiment."""
    if skip_if_done and is_done(cfg.name):
        print(f'[SKIP] {cfg.name} already in results.json')
        return None

    print(f"\n{'='*60}")
    print(f'Running: {cfg.name}')
    print(f"{'='*60}")

    metrics = train_experiment(cfg, device=DEVICE)

    if metrics.get('status') == 'crash_nan':
        log_result(cfg, metrics, status='crash_nan', conclusion='NaN/Inf loss during training')
        return metrics

    # Generate Diagnostics
    val_gen = make_torch_loader(cfg.benchmark, 'val', 8, device=DEVICE)
    x_v, y_v = next(val_gen)
    with torch.no_grad():
        y_p = metrics['model'](x_v)
    spec_bias = calculate_spectral_bias(y_p.cpu().numpy(), y_v.cpu().numpy())
    metrics['spec_bias'] = spec_bias

    exp_id = log_result(cfg, metrics)
    generate_inspect_png(exp_id, x_v.cpu().numpy(), y_v.cpu().numpy(), y_p.cpu().numpy(), cfg.benchmark)

    # Quick loss curve
    fig, ax = plt.subplots(figsize=(8, 3))
    h = metrics['loss_history']
    # smooth
    w = 50
    if len(h) > w:
        smoothed = np.convolve(h, np.ones(w)/w, mode='valid')
        ax.plot(smoothed, color='steelblue', lw=1.5, label='train loss (smoothed)')
    else:
        ax.plot(h, color='steelblue', lw=1.5, label='train loss')
    ax.set_title(f'{cfg.name}  val_l2_rel={metrics["val_l2_rel"]:.4f}')
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.set_yscale('log')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return metrics


print('run_and_log() ready')

## Cell 11 — SOTA reference table

In [ ]:
SOTA = {
    'burgers_1d':   0.0149,
    'kdv_1d':       0.010,
    'wave_1d':      0.005,
    'darcy_2d': 0.0108,
    'ns_2d':    0.0128,
}

OUR_BEST_MLX = {
    'burgers_1d':   0.1468,
    'kdv_1d':       0.0020,
    'wave_1d':      0.000992,
    'darcy_2d': 0.1041,
    'ns_2d':    0.01428,
}

print(f"{'Benchmark':<18} {'SOTA':>8} {'Our best (M1)':>14}")
print('-' * 44)
for bm, sota in SOTA.items():
    best = OUR_BEST_MLX.get(bm, float('nan'))
    gap  = best / sota if sota > 0 else float('nan')
    print(f'{bm:<18} {sota:>8.4f} {best:>14.4f}  ({gap:.1f}x gap)')

## Cell 12 — Dashboard results view

In [ ]:
def show_results(benchmark=None):
    experiments = _load_results()
    if benchmark:
        experiments = [e for e in experiments if e['benchmark'] == benchmark]
    if not experiments:
        print('No results yet.')
        return

    # Sort by val_l2_rel
    experiments.sort(key=lambda e: e.get('val_l2_rel', 999))

    # Group by benchmark
    from itertools import groupby
    by_bm = {}
    for e in experiments:
        by_bm.setdefault(e['benchmark'], []).append(e)

    for bm, rows in by_bm.items():
        sota = SOTA.get(bm, None)
        print(f"\n{'='*70}")
        print(f'  {bm}   (SOTA={sota})')
        print(f"  {'Model':<12} {'val_l2_rel':>12} {'gap':>8} {'status':>8} {'framework':>12}")
        print(f"  {'-'*56}")
        for e in rows[:10]:
            val  = e.get('val_l2_rel', float('nan'))
            gap  = f'{val/sota:.1f}x' if sota else '—'
            fw   = e.get('diag', {}).get('framework', 'mlx')
            print(f"  {e['model']:<12} {val:>12.6f} {gap:>8} {e.get('status','keep'):>8} {fw:>12}")

show_results()

## Cell 13 — Sync results to local repo (git commit)

Run this after experiments to commit results back to the repo so the dashboard can read them.

In [ ]:
def git_commit_results(message=None):
    """
    Stage results.json + results.tsv + logs/ and commit to the repo.
    On Colab, you'll need to `git push` separately after downloading credentials.
    """
    if not (REPO_PATH / '.git').exists():
        print('Not a git repo — skipping commit. Copy results.json manually.')
        return

    msg = message or f'feat(colab): add {DEVICE} experiment results'
    try:
        _sp.run(['git', '-C', str(REPO_PATH), 'add',
                 'results.json', 'results.tsv'], check=True)
        log_files = list(LOGS_DIR.glob('*.log'))
        if log_files:
            _sp.run(['git', '-C', str(REPO_PATH), 'add', 'logs/'], check=True)
        _sp.run(['git', '-C', str(REPO_PATH), 'commit', '-m', msg], check=True)
        print(f'Committed: {msg}')
    except _sp.CalledProcessError as e:
        print(f'Git error: {e}')

# git_commit_results()  # Uncomment after experiments
print('git_commit_results() ready')

---
## Run Experiments

Edit the config below and run the cell to start training.

### Example A — Burgers 1D: baseline FNO (replicate MLX best)

In [ ]:
cfg_a = ExperimentConfig(
    name        = 'colab_fno_burgers_h128_l8_m24_aug',
    benchmark   = 'burgers_1d',
    model       = 'FNO',
    hidden_dim  = 128,
    n_layers    = 8,
    n_modes     = 24,
    augment     = True,
    budget_s    = 300,
    priority    = 1,
    rationale   = 'Baseline FNO replication on Colab GPU',
)

result_a = run_and_log(cfg_a)

### Example B — KdV 1D: RFNO (MLX SOTA is 0.0020)

In [ ]:
cfg_b = ExperimentConfig(
    name        = 'colab_rfno_kdv_h128_l8_m24',
    benchmark   = 'kdv_1d',
    model       = 'RFNO',
    hidden_dim  = 128,
    n_layers    = 8,
    n_modes     = 24,
    budget_s    = 300,
    priority    = 1,
    rationale   = 'RFNO on KdV — best on MLX; test if CUDA gives more steps',
)

result_b = run_and_log(cfg_b)

### Example C — Wave 1D: FNO small (MLX best: 0.000992)

In [ ]:
cfg_c = ExperimentConfig(
    name        = 'colab_fno_wave_h64_l4_m16',
    benchmark   = 'wave_1d',
    model       = 'FNO',
    hidden_dim  = 64,
    n_layers    = 4,
    n_modes     = 16,
    budget_s    = 300,
    priority    = 1,
    rationale   = 'Small FNO on Wave — budget-dominated; CUDA allows many more steps',
)

result_c = run_and_log(cfg_c)

### Example D — NS 2D fix: FNO2d with extended budget (600s)

In [ ]:
# Note: 2D constraints are different on CUDA (no Apple Silicon OOM issues)
# Start conservative then scale up
cfg_d = ExperimentConfig(
    name        = 'colab_fno2d_ns_h64_l4_m8_600s',
    benchmark   = 'ns_2d',
    model       = 'FNO',
    hidden_dim  = 64,
    n_layers    = 4,
    n_modes     = 8,
    budget_s    = 600,
    priority    = 1,
    rationale   = 'FNO2d on NS — CUDA allows h=64 without OOM; 600s budget',
)

result_d = run_and_log(cfg_d)

### Batch runner — run a list of configs

In [ ]:
# Define your batch here
batch_configs = [
    ExperimentConfig(
        name='colab_fno_burgers_h256_l8_m24',
        benchmark='burgers_1d', model='FNO',
        hidden_dim=256, n_layers=8, n_modes=24, augment=True,
        budget_s=300, priority=2,
        rationale='Wide FNO — more CUDA memory allows h=256 without step-time penalty',
    ),
    ExperimentConfig(
        name='colab_rfno_burgers_h128_l10_m24',
        benchmark='burgers_1d', model='RFNO',
        hidden_dim=128, n_layers=10, n_modes=24, augment=True,
        budget_s=300, priority=2,
        rationale='Deep RFNO on Burgers — pre-LN should handle l=10 depth',
    ),
    ExperimentConfig(
        name='colab_uno_burgers_h128_l4_m24',
        benchmark='burgers_1d', model='UNO',
        hidden_dim=128, n_layers=4, n_modes=24, augment=True,
        budget_s=300, priority=2,
        rationale='UNO multi-scale architecture on Burgers',
    ),
    ExperimentConfig(
        name='colab_rfno_kdv_h256_l8_m24',
        benchmark='kdv_1d', model='RFNO',
        hidden_dim=256, n_layers=8, n_modes=24,
        budget_s=300, priority=2,
        rationale='Wide RFNO on KdV — CUDA vs MLX: h=256 step-time comparison',
    ),
]

print(f'Batch of {len(batch_configs)} experiments queued')
for cfg in batch_configs:
    print(f'  {cfg.name}')

# Uncomment to run:
# for cfg in batch_configs:
#     run_and_log(cfg)

### View results after running

In [ ]:
show_results()

### Commit results back to repo

In [ ]:
# git_commit_results('feat(colab): GPU experiment results')

---
## Notes

### Colab workflow
1. Clone the repo at the top (`REPO_PATH`) so `results.json` is shared.
2. Run experiments (examples above or add your own `ExperimentConfig`).
3. Call `git_commit_results()` then `git push` to sync back.
4. Open `ui/dashboard.html` (pointing at the local FastAPI server) to see Colab results alongside MLX results.

### 2D benchmarks on CUDA
On a T4/A100 you are **not** limited to `h≤32`. Start at `h=64, l=4, m=12` and scale up. The Apple Silicon constraints (`h≤32`) do not apply here.

### RFNO is 1D-only (same as MLX version)
The RFNO reshape logic breaks on 2D inputs — use `FNO` (`FNO2d`) for 2D benchmarks.

### Benchmarks supported by this notebook
| benchmark | solver | notes |
|---|---|---|
| `burgers_1d` | IMEX-Euler | primary target |
| `kdv_1d` | ETDRK4 | soliton dynamics |
| `wave_1d` | Störmer-Verlet | very easy for FNO |
| `darcy_2d` | PCG (corrected) | elliptic |
| `ns_2d` | semi-implicit spectral | CFL-safe |

Euler 1D, SWE 2D, Allen-Cahn 2D require the simulation modules from the MLX repo — add them by copying `simulations/` and adapting the loader.

## Cell 14 — Queue Reader (parses experiments.yaml)

In [ ]:
import yaml

# GPU budget floors — lower than MLX because CUDA is faster per-step
BUDGET_FLOOR_GPU_1D = 600   # 10 min minimum for 1D benchmarks
BUDGET_FLOOR_GPU_2D = 1200  # 20 min minimum for 2D benchmarks

BENCHMARKS_2D = {'darcy_2d', 'ns_2d', 'swe_2d', 'allen_cahn_2d', 'ns_hre'}

# Models this notebook can train (PyTorch implementations available)
SUPPORTED_MODELS = {'FNO', 'FNO1D', 'RFNO', 'RFNO1D', 'UNO', 'FNO2D', 'FNO2d', 'RFNO2D'}

def load_yaml_queue(yaml_path=None):
    """Load all experiments from experiments.yaml as ExperimentConfig objects."""
    path = Path(yaml_path) if yaml_path else REPO_PATH / 'experiments.yaml'
    if not path.exists():
        print(f'experiments.yaml not found at {path}')
        return []
    raw = yaml.safe_load(path.read_text()) or []
    configs = []
    for entry in raw:
        try:
            cfg = ExperimentConfig(
                name        = entry['name'],
                benchmark   = entry['benchmark'],
                model       = entry['model'],
                hidden_dim  = int(entry.get('hidden_dim', 64)),
                n_layers    = int(entry.get('n_layers', 4)),
                n_modes     = int(entry.get('n_modes', 16)),
                n_levels    = int(entry.get('n_levels', 3)),
                n_head      = int(entry.get('n_head', 4)),
                slice_num   = int(entry.get('slice_num', 32)),
                lr          = float(entry.get('lr', 1e-3)),
                batch_size  = int(entry.get('batch_size', 64)),
                grad_clip   = float(entry.get('grad_clip', 1.0)),
                loss_type   = entry.get('loss_type', entry.get('loss', 'l2_rel')),
                h1_alpha    = float(entry.get('h1_alpha', 0.1)),
                augment     = bool(entry.get('augment', False)),
                budget_s    = int(entry.get('budget_s', 300)),
                parent_name = entry.get('parent_name', ''),
                priority    = int(entry.get('priority', 5)),
                rationale   = entry.get('rationale', ''),
                expected    = entry.get('expected', ''),
                paper_ref   = entry.get('paper_ref', ''),
            )
            configs.append(cfg)
        except (KeyError, TypeError) as e:
            print(f'  [WARN] Skipping malformed entry: {entry.get("name","?")} — {e}')
    return configs


def get_pending_queue(priority_max=None, benchmark_filter=None, model_filter=None,
                      supported_only=True, yaml_path=None):
    """Return sorted pending experiments not yet in results.json.

    Args:
        priority_max:     only include experiments with priority <= this value
        benchmark_filter: filter to a specific benchmark string
        model_filter:     filter to a specific model string (case-insensitive)
        supported_only:   skip models without PyTorch implementations
    """
    all_cfgs = load_yaml_queue(yaml_path)
    done     = set(e.get('config', {}).get('name') for e in _load_results())

    pending = []
    skipped_unsupported = []
    for cfg in all_cfgs:
        if cfg.name in done:
            continue
        if benchmark_filter and cfg.benchmark != benchmark_filter:
            continue
        if model_filter and cfg.model.upper() != model_filter.upper():
            continue
        if priority_max is not None and cfg.priority > priority_max:
            continue
        model_key = cfg.model.upper().replace('2D', '').replace('1D', '')
        if supported_only and cfg.model.upper() not in SUPPORTED_MODELS:
            skipped_unsupported.append(cfg.name)
            continue
        pending.append(cfg)

    pending.sort(key=lambda c: c.priority)

    if skipped_unsupported:
        print(f'  [INFO] Skipped {len(skipped_unsupported)} experiments with unsupported models '
              f'(no PyTorch impl): {set(c.model for c in load_yaml_queue(yaml_path) if c.name in set(skipped_unsupported))}')
    return pending


def apply_gpu_budget_floor(cfg):
    """Raise budget_s to GPU-appropriate minimums (lower than MLX floors)."""
    import dataclasses
    floor = BUDGET_FLOOR_GPU_2D if cfg.benchmark in BENCHMARKS_2D else BUDGET_FLOOR_GPU_1D
    if cfg.budget_s < floor:
        print(f'  [BUDGET] {cfg.name}: {cfg.budget_s}s → {floor}s (GPU floor)')
        return dataclasses.replace(cfg, budget_s=floor)
    return cfg


def preview_queue(priority_max=None, benchmark_filter=None):
    """Print a human-readable queue summary."""
    queue = get_pending_queue(priority_max=priority_max, benchmark_filter=benchmark_filter)
    total = load_yaml_queue()
    done  = set(e.get('config', {}).get('name') for e in _load_results())
    print(f'Queue summary: {len(queue)} pending / {len(total)} total / {len(done)} done')
    print(f"{'#':<4} {'Pri':<4} {'Benchmark':<18} {'Model':<16} {'Name'}")
    print('-' * 80)
    for i, cfg in enumerate(queue[:30], 1):
        print(f'{i:<4} {cfg.priority:<4} {cfg.benchmark:<18} {cfg.model:<16} {cfg.name}')
    if len(queue) > 30:
        print(f'  ... and {len(queue)-30} more')
    return queue


preview_queue()

## Cell 15 — Trajectories Logger (RL replay buffer, compatible with MLX harness)

In [ ]:
from datetime import datetime as _dt

TRAJECTORIES_FILE = LOGS_DIR / 'trajectories.jsonl'

def write_trajectory(entry: dict) -> None:
    """Append one JSON line to logs/trajectories.jsonl.
    Compatible with the MLX harness replay buffer format.
    Fields: timestamp, benchmark, state, hypothesis, action,
            expected_outcome, outcome, diag_snapshot, device, framework.
    """
    try:
        LOGS_DIR.mkdir(parents=True, exist_ok=True)
        if 'timestamp' not in entry:
            entry['timestamp'] = _dt.utcnow().isoformat() + 'Z'
        entry.setdefault('device', DEVICE)
        entry.setdefault('framework', 'pytorch')
        with open(TRAJECTORIES_FILE, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    except Exception as e:
        print(f'[WARN] trajectory write failed: {e}')


def read_trajectories(benchmark=None, last_n=20):
    """Read recent trajectory entries, optionally filtered by benchmark."""
    if not TRAJECTORIES_FILE.exists():
        return []
    lines = TRAJECTORIES_FILE.read_text().strip().split('\n')
    entries = []
    for line in lines:
        try:
            e = json.loads(line)
            if benchmark is None or e.get('benchmark') == benchmark:
                entries.append(e)
        except json.JSONDecodeError:
            pass
    return entries[-last_n:]


print('Trajectories logger ready →', TRAJECTORIES_FILE)

## Cell 16 — Critique & Guidelines Engine

After each run the autonomous loop calls `critique_run()` to:
- Compare result vs SOTA and our current best
- Identify spectral bias, NaN patterns, plateau signals
- Print actionable next-step guidelines
- Write a structured critique to `trajectories.jsonl`

In [ ]:
def _current_best(benchmark):
    """Return best val_l2_rel seen so far for this benchmark across all results."""
    kept = [e['val_l2_rel'] for e in _load_results()
            if e.get('benchmark') == benchmark
            and e.get('status') not in ('crash_nan', 'discard')
            and e.get('val_l2_rel') is not None]
    return min(kept) if kept else float('inf')


def critique_run(cfg, metrics, exp_id=None):
    """
    Analyse a completed run and print structured critique + guidelines.
    Returns a dict with the critique summary (also written to trajectories.jsonl).
    """
    val        = metrics.get('val_l2_rel', float('nan'))
    sota       = SOTA.get(cfg.benchmark)
    our_best   = _current_best(cfg.benchmark)
    mlx_best   = OUR_BEST_MLX.get(cfg.benchmark)
    spec_bias  = metrics.get('spec_bias', {})
    status     = metrics.get('status', 'unknown')
    steps      = metrics.get('num_steps', 0)
    budget     = cfg.budget_s
    training_s = metrics.get('training_s', 0)

    lines = []
    lines.append(f'\n{"─"*70}')
    lines.append(f'  CRITIQUE: {cfg.name}')
    lines.append(f'{"─"*70}')

    # ── Result classification ──────────────────────────────────────────────
    if status == 'crash_nan':
        verdict = 'CRASH'
        lines.append('  ❌ CRASH: NaN/Inf loss during training.')
        lines.append('  CAUSE: likely LR too high, bad initialisation, or numerical instability.')
        lines.append('  FIX:  ① lower lr by 5×  ② add grad_clip=0.5  ③ switch loss to l2_rel')
    elif math.isnan(val):
        verdict = 'INVALID'
        lines.append('  ❌ INVALID result (NaN val).')
    else:
        gap_sota = val / sota if sota else float('nan')
        gap_best = val / our_best if our_best < float('inf') else float('nan')
        gap_mlx  = val / mlx_best if mlx_best else float('nan')

        if gap_sota <= 1.0:
            verdict = 'SOTA_MATCH'
            lines.append(f'  🏆 SOTA MATCHED or beaten: val={val:.6f} vs SOTA={sota:.6f} ({gap_sota:.2f}×)')
        elif gap_sota <= 1.5:
            verdict = 'NEAR_SOTA'
            lines.append(f'  ✅ Near-SOTA: val={val:.6f}  gap={gap_sota:.2f}× SOTA={sota:.6f}')
        elif gap_sota <= 3.0:
            verdict = 'PROMISING'
            lines.append(f'  🔶 Promising: val={val:.6f}  gap={gap_sota:.2f}× SOTA={sota:.6f}')
        else:
            verdict = 'UNDERPERFORMING'
            lines.append(f'  🔴 Under-performing: val={val:.6f}  gap={gap_sota:.2f}× SOTA={sota:.6f}')

        lines.append(f'  vs our best (all devices): {our_best:.6f}  vs MLX best: {mlx_best}')

        # New best?
        if val < our_best * 0.995 and not math.isnan(val):
            lines.append('  ⭐ NEW BEST across all devices!')

        # GPU advantage check
        if mlx_best and not math.isnan(gap_mlx):
            if val < mlx_best * 0.98:
                lines.append(f'  📈 GPU advantage confirmed: {gap_mlx:.2f}× better than M1 best')
            elif val > mlx_best * 1.05:
                lines.append(f'  📉 GPU slower than M1 ({gap_mlx:.2f}×) — check batch size / budget')

    # ── Budget efficiency ──────────────────────────────────────────────────
    if steps > 0 and training_s > 0:
        steps_per_sec = steps / training_s
        budget_used   = training_s / budget
        lines.append(f'  Budget: {training_s:.0f}s / {budget}s used ({budget_used*100:.0f}%)  '
                     f'{steps_per_sec:.1f} steps/s  {steps} total steps')
        if budget_used < 0.9:
            lines.append('  ⚠️  Budget under-utilised — training ended early (early-stop or crash)')
        if steps_per_sec > 50 and budget <= 600:
            lines.append('  💡 High step rate — consider extending budget_s for this benchmark on GPU')

    # ── Spectral bias analysis ─────────────────────────────────────────────
    if spec_bias:
        lo  = spec_bias.get('low_freq_error', 0)
        hi  = spec_bias.get('high_freq_error', 0)
        gap = spec_bias.get('spectral_gap', 0)
        lines.append(f'  Spectral bias: low={lo:.4f}  high={hi:.4f}  gap={gap:.4f}')
        if hi > 0.3:
            lines.append('  🔵 High-frequency under-prediction detected.')
            lines.append('  FIX: try loss=spectral or increase n_modes')
        if lo > hi * 3:
            lines.append('  🔵 Low-freq dominated error — may need deeper network or more modes')

    # ── Architecture guidelines ────────────────────────────────────────────
    lines.append('')
    lines.append('  GUIDELINES for next experiments on this benchmark:')
    bm = cfg.benchmark
    if verdict in ('CRASH', 'UNDERPERFORMING'):
        lines.append(f'  • Try h={max(32, cfg.hidden_dim//2)}, l={cfg.n_layers}, m={cfg.n_modes} (halve width first)')
        lines.append(f'  • Or switch to RFNO (pre-LN residual — more stable for deep nets)')
        if 'ns_2d' in bm or 'darcy' in bm:
            lines.append(f'  • 2D: ensure h≤64 on T4, ≤128 on A100 to avoid OOM')
    elif verdict == 'SOTA_MATCH':
        lines.append(f'  • Push further: try h={cfg.hidden_dim*2}, same l/m')
        lines.append(f'  • Or try UNO (multi-scale) for additional gains')
        lines.append(f'  • Add --augment=True if not already set')
    elif verdict in ('PROMISING', 'NEAR_SOTA'):
        lines.append(f'  • Scale up: h={cfg.hidden_dim*2} or l={cfg.n_layers+2}')
        lines.append(f'  • Extend budget: budget_s={cfg.budget_s*2}s for more steps')
        if cfg.loss_type == 'l2_rel' and ('burgers' in bm or 'kdv' in bm):
            lines.append(f'  • Try loss=h1 or loss=h1_adaptive (gradient penalty helps on shocks)')

    # GPU-specific opportunities (not possible on M1)
    lines.append('')
    lines.append('  GPU OPPORTUNITIES (not limited by Apple Silicon constraints):')
    if cfg.hidden_dim <= 64:
        lines.append(f'  • Scale to h=128 or h=256 — no MLX memory constraint here')
    if bm in BENCHMARKS_2D and cfg.hidden_dim <= 32:
        lines.append(f'  • 2D benchmark: h=64 or h=128 is safe on T4; h=256 on A100')
    if cfg.n_layers <= 4:
        lines.append(f'  • Try l=8 — deeper nets benefit from CUDA parallel execution')

    lines.append(f'{"─"*70}\n')

    for line in lines:
        print(line)

    critique_summary = {
        'verdict':       verdict,
        'val_l2_rel':    val,
        'sota_gap':      round(val / sota, 4) if sota and not math.isnan(val) else None,
        'new_best':      (val < our_best * 0.995) if not math.isnan(val) else False,
        'spec_bias':     spec_bias,
        'steps':         steps,
        'steps_per_sec': round(steps / training_s, 2) if training_s > 0 else 0,
    }

    write_trajectory({
        'benchmark':        cfg.benchmark,
        'state':            'completed',
        'hypothesis':       cfg.rationale or cfg.model,
        'action':           cfg.name,
        'expected_outcome': cfg.expected or f'val < {our_best:.4f}',
        'outcome':          json.dumps(critique_summary),
        'diag_snapshot':    critique_summary,
        'exp_id':           exp_id,
    })

    return critique_summary


print('Critique engine ready')

## Cell 17 — SOTA Gap Analysis & Auto-Suggest

In [ ]:
def sota_gap_analysis():
    """Print a full SOTA gap analysis across all benchmarks, then suggest next actions."""
    results    = _load_results()
    queue      = load_yaml_queue()
    done_names = set(e.get('config', {}).get('name') for e in results)

    print(f'\n{"═"*72}')
    print('  SOTA GAP ANALYSIS')
    print(f'{"═"*72}')
    print(f'  {"Benchmark":<18} {"SOTA":>8} {"Best(GPU)":>10} {"Best(M1)":>10} {"Gap":>7} {"Status"}')
    print(f'  {"─"*68}')

    ranked = []
    for bm, sota in sorted(SOTA.items()):
        gpu_results = [e for e in results
                       if e.get('benchmark') == bm
                       and e.get('status') not in ('crash_nan', 'discard')
                       and e.get('val_l2_rel') is not None
                       and e.get('diag', {}).get('framework') == 'pytorch']
        gpu_best   = min((e['val_l2_rel'] for e in gpu_results), default=None)
        mlx_best   = OUR_BEST_MLX.get(bm)
        all_best   = min(filter(None, [gpu_best, mlx_best]), default=None)
        gap        = (all_best / sota) if all_best and sota else float('inf')

        if gap <= 1.0:
            status = '🏆 SOTA'
        elif gap <= 1.5:
            status = '✅ Near-SOTA'
        elif gap <= 3.0:
            status = '🔶 Promising'
        elif gap < float('inf'):
            status = '🔴 Far'
        else:
            status = '⬜ No GPU run'

        gpu_str = f'{gpu_best:.5f}' if gpu_best else '—'
        mlx_str = f'{mlx_best:.5f}' if mlx_best else '—'
        gap_str = f'{gap:.1f}×' if gap < float('inf') else '—'
        print(f'  {bm:<18} {sota:>8.4f} {gpu_str:>10} {mlx_str:>10} {gap_str:>7}  {status}')
        ranked.append((gap, bm))

    ranked.sort(reverse=True)

    pending_support = get_pending_queue(supported_only=True)
    pending_all     = get_pending_queue(supported_only=False)
    print(f'\n  Queue: {len(pending_support)} runnable / {len(pending_all)} total pending')

    print(f'\n{"─"*72}')
    print('  AUTO-SUGGEST: Top 5 recommended next experiments')
    print(f'{"─"*72}')

    suggestions = []

    for gap, bm in ranked:
        bm_done = [e for e in results if e.get('benchmark') == bm
                   and e.get('diag', {}).get('framework') == 'pytorch']
        if not bm_done and gap > 1.0:
            suggestions.append({
                'priority': 'HIGH',
                'reason':   f'No GPU run yet for {bm} (gap={gap:.1f}×)',
                'action':   f'Run FNO baseline on {bm} — establish GPU baseline',
                'config':   f'model=FNO, h=128, l=8, m=24, budget=1200s',
            })
            if len(suggestions) >= 2:
                break

    for e in sorted(results, key=lambda x: x.get('val_l2_rel', 999)):
        bm   = e.get('benchmark')
        sota = SOTA.get(bm, float('inf'))
        val  = e.get('val_l2_rel', float('inf'))
        if sota and 1.0 < val / sota <= 2.5 and e.get('diag', {}).get('framework') == 'pytorch':
            cfg_data = e.get('config', {})
            h        = cfg_data.get('hidden_dim', 64)
            l        = cfg_data.get('n_layers', 4)
            m        = cfg_data.get('n_modes', 16)
            suggestions.append({
                'priority': 'MEDIUM',
                'reason':   f'{bm}: val={val:.4f} ({val/sota:.1f}× SOTA) — scale-up candidate',
                'action':   f'Double width: h={h*2}, l={l}, m={m}',
                'config':   f'model={cfg_data.get("model","FNO")}, h={h*2}, l={l}, m={m}, budget={cfg_data.get("budget_s",600)*2}s',
            })
            if len(suggestions) >= 5:
                break

    for cfg in pending_support[:max(0, 5 - len(suggestions))]:
        suggestions.append({
            'priority': 'QUEUE',
            'reason':   cfg.rationale or f'Queue priority={cfg.priority}',
            'action':   f'Run {cfg.name}',
            'config':   f'model={cfg.model}, h={cfg.hidden_dim}, l={cfg.n_layers}, budget={cfg.budget_s}s',
        })

    for i, s in enumerate(suggestions[:5], 1):
        print(f'\n  [{i}] [{s["priority"]}] {s["reason"]}')
        print(f'      Action: {s["action"]}')
        print(f'      Config: {s["config"]}')

    print(f'\n{"═"*72}\n')
    return suggestions


sota_gap_analysis()

## Cell 18 — Autonomous Queue Runner

Reads `experiments.yaml`, runs all pending supported experiments in priority order,
applies GPU budget floors, critiques each result, and logs to `trajectories.jsonl`.

**Differences from MLX `autorun.py`:**
- Uses CUDA/GPU instead of MLX / Apple Silicon
- Budget floors: 600 s (1D) / 1200 s (2D) instead of 1800 / 3600 s
- No Apple Silicon memory constraints (h≤32 rule does not apply)
- Models without PyTorch implementations are skipped with a warning
- No `--auto` HPO fallback (add configs manually or via `sota_gap_analysis()`)

**Usage:**
```python
run_queue()                                          # run all pending
run_queue(priority_max=2)                            # high-priority only
run_queue(benchmark_filter='burgers_1d')             # one benchmark
run_queue(dry_run=True)                              # print plan only
run_queue(max_experiments=10, max_time_s=3600)       # capped run
```

In [ ]:
def run_queue(
    priority_max    = None,
    benchmark_filter = None,
    model_filter    = None,
    dry_run         = False,
    max_experiments = None,
    max_time_s      = None,
    skip_if_done    = True,
    commit_results  = False,
):
    """
    Autonomous experiment runner — reads experiments.yaml, runs all pending
    supported experiments, critiques each result, and logs everything.

    Colab-GPU equivalent of: uv run autorun.py --auto --commit
    """
    queue = get_pending_queue(
        priority_max=priority_max,
        benchmark_filter=benchmark_filter,
        model_filter=model_filter,
        supported_only=True,
    )

    if not queue:
        print('Queue is empty — nothing to run.')
        print('Call sota_gap_analysis() for suggestions or add entries to experiments.yaml.')
        return

    print(f'\n{"═"*72}')
    print(f'  AUTONOMOUS QUEUE RUNNER  ({DEVICE.upper()})')
    print(f'{"═"*72}')
    print(f'  {len(queue)} experiments queued')
    if priority_max:    print(f'  Priority filter: ≤ {priority_max}')
    if benchmark_filter: print(f'  Benchmark filter: {benchmark_filter}')
    if max_experiments: print(f'  Max experiments: {max_experiments}')
    if max_time_s:      print(f'  Max wall time:   {max_time_s}s ({max_time_s/3600:.1f}h)')
    print()

    if dry_run:
        print('  DRY RUN — experiments that would run:')
        for i, cfg in enumerate(queue, 1):
            floored = apply_gpu_budget_floor(cfg)
            print(f'  {i:3d}. [{cfg.priority}] {cfg.benchmark:<18} {cfg.model:<14} '
                  f'h={cfg.hidden_dim} l={cfg.n_layers} m={cfg.n_modes} '
                  f'budget={floored.budget_s}s  {cfg.name}')
        total_h = sum(apply_gpu_budget_floor(c).budget_s for c in queue) / 3600
        print(f'\n  Total budget (sequential): {total_h:.1f}h')
        return

    loop_start = time.time()
    n_ran = n_new_best = n_crashed = 0
    session_log = []

    write_trajectory({
        'benchmark': 'ALL', 'state': 'session_start',
        'action': f'run_queue priority_max={priority_max} bm={benchmark_filter}',
        'expected_outcome': f'run {len(queue)} experiments on {DEVICE}',
    })

    for i, cfg in enumerate(queue):
        if max_time_s and (time.time() - loop_start) >= max_time_s:
            print(f'\n[STOP] Wall time limit reached ({max_time_s}s). Ran {n_ran}.')
            break
        if max_experiments and n_ran >= max_experiments:
            print(f'\n[STOP] Experiment limit reached ({max_experiments}). Done.')
            break
        if skip_if_done and is_done(cfg.name):
            print(f'[SKIP] {cfg.name}')
            continue

        cfg = apply_gpu_budget_floor(cfg)
        elapsed_h = (time.time() - loop_start) / 3600
        print(f'\n[{i+1}/{len(queue)}] elapsed={elapsed_h:.1f}h  new_bests={n_new_best}  crashes={n_crashed}')

        write_trajectory({
            'benchmark': cfg.benchmark, 'state': 'queued',
            'hypothesis': cfg.rationale or cfg.model, 'action': f'queue {cfg.name}',
            'expected_outcome': cfg.expected or f'val < {_current_best(cfg.benchmark):.4f}',
        })

        metrics = train_experiment(cfg, device=DEVICE)

        # Spectral diagnostics
        if metrics.get('status') != 'crash_nan':
            try:
                val_gen  = make_torch_loader(cfg.benchmark, 'val', 8, device=DEVICE)
                x_v, y_v = next(val_gen)
                with torch.no_grad():
                    y_p = metrics['model'](x_v)
                metrics['spec_bias'] = calculate_spectral_bias(
                    y_p.cpu().numpy(), y_v.cpu().numpy())
            except Exception as e:
                print(f'  [WARN] spectral bias: {e}')

        status = 'crash_nan' if metrics.get('status') == 'crash_nan' else 'keep'
        exp_id = log_result(cfg, metrics, status=status)

        # Inspect PNG
        if status == 'keep':
            try:
                val_gen  = make_torch_loader(cfg.benchmark, 'val', 8, device=DEVICE)
                x_v, y_v = next(val_gen)
                with torch.no_grad():
                    y_p = metrics['model'](x_v)
                generate_inspect_png(exp_id,
                    x_v.cpu().numpy(), y_v.cpu().numpy(), y_p.cpu().numpy(), cfg.benchmark)
            except Exception as e:
                print(f'  [WARN] inspect PNG: {e}')

        critique = critique_run(cfg, metrics, exp_id=exp_id)

        n_ran += 1
        if status == 'crash_nan':
            n_crashed += 1
        elif critique.get('new_best'):
            n_new_best += 1

        session_log.append({
            'name': cfg.name, 'benchmark': cfg.benchmark, 'model': cfg.model,
            'val': metrics.get('val_l2_rel'), 'verdict': critique.get('verdict'),
            'new_best': critique.get('new_best', False),
        })

        if commit_results and status == 'keep':
            git_commit_results(
                f'feat(colab/{DEVICE}): {cfg.benchmark} {cfg.model} '
                f'val={metrics["val_l2_rel"]:.4f}')

        # Loss curve
        if metrics.get('loss_history'):
            try:
                import matplotlib.pyplot as plt
                fig, ax = plt.subplots(figsize=(8, 3))
                h = metrics['loss_history']
                w = min(50, len(h))
                smoothed = np.convolve(h, np.ones(w)/w, mode='valid') if len(h) > w else h
                ax.plot(smoothed, color='steelblue', lw=1.5)
                ax.set_title(f'{cfg.name}  val={metrics.get("val_l2_rel", "?"):.4f}  '
                             f'[{critique["verdict"]}]')
                ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.set_yscale('log')
                plt.tight_layout(); plt.show()
            except Exception:
                pass

    # Session summary
    total_s = time.time() - loop_start
    print(f'\n{"═"*72}')
    print(f'  SESSION COMPLETE  —  {n_ran} ran  {n_new_best} new bests  '
          f'{n_crashed} crashes  {total_s/3600:.2f}h')
    print(f'{"═"*72}')
    print(f'  {"Name":<42} {"Benchmark":<15} {"Val":>10}  {"Verdict"}')
    print(f'  {"─"*78}')
    for row in session_log:
        val_s = f'{row["val"]:.6f}' if row['val'] is not None else '—'
        star  = ' ⭐' if row['new_best'] else ''
        print(f'  {row["name"]:<42} {row["benchmark"]:<15} {val_s:>10}  {row["verdict"]}{star}')
    print(f'{"═"*72}\n')

    write_trajectory({
        'benchmark': 'ALL', 'state': 'session_end',
        'action': f'completed {n_ran} experiments',
        'outcome': json.dumps({'n_ran': n_ran, 'n_new_best': n_new_best,
                               'n_crashed': n_crashed, 'wall_s': round(total_s)}),
    })

    sota_gap_analysis()
    return session_log


print('run_queue() ready')
print()
print('  run_queue(dry_run=True)             # preview plan')
print('  run_queue(priority_max=2)           # run high-priority only')
print('  run_queue(max_time_s=3600)          # run for up to 1 hour')
print('  run_queue(benchmark_filter="ns_2d") # single benchmark')
print('  run_queue(commit_results=True)      # auto-commit after each run')

## Cell 19 — Launch Autonomous Run

Run all cells above (Cells 1–18), then execute one of these:

In [ ]:
# ── Option 1: Preview the full queue (no training) ────────────────────────────
run_queue(dry_run=True)

In [ ]:
# ── Option 2: Run high-priority experiments (priority ≤ 2) ───────────────────
# run_queue(priority_max=2, commit_results=True)

# ── Option 3: Full autonomous run (all supported, time-capped at 4 h) ────────
# run_queue(max_time_s=4*3600, commit_results=True)

# ── Option 4: Focus on the benchmark with the biggest SOTA gap ───────────────
# run_queue(benchmark_filter='burgers_1d', max_time_s=3600)